# Connecting to Neo4j
The following cells connect to the KG database and run some queries on the KG.
It's just here to connect and test connection to the KG.

In [ ]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 2.2 MB/s eta 0:00:00


In [ ]:
import csv
from neo4j import GraphDatabase

In [11]:
# for rendering on github:
from google.colab import output
output.clear()

In [1]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

uri = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
username = os.getenv("NEO4J_USERNAME", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "password")

username, password

('neo4j', 'SnuqfMR_dIqDv1smxBBcQJyzN_bG6YLEdzNf_GFD1_U')

In [ ]:
try:
    driver = GraphDatabase.driver(uri, auth=(username, password))
    driver.verify_connectivity()
    print("Connection successful!", flush=True)
except Exception as e:
    print(f"Connection failed: {e}", flush=True)
    exit(1)

def test_connection():
    try:
        with driver.session() as session:
            result = session.run("MATCH (n) RETURN count(n) AS c")
            print("Number of nodes in the graph:", result.single()["c"])
    except Exception as e:
        print("Connection failed:", e)

test_connection()

Connection successful!
Number of nodes in the graph: 7944


# Intent classification
## Intent Definition

First, we define intents based on the existing data in the Knowledge Graph.

In [ ]:
# For simple, rule-based classification we use two signal keywords.
# The 'any' keywords are indicators of this intent, for each word from any in the input text, a score 1 is added to the total score.
# The exclude keywords are there to disqualify an intent.
# The intent with the highest score is chosen.

import re

# Intent classification rules with scoring
INTENT_RULES = [

    # 1. route_operations_query
    {
        "intent": "route_operations_query",
        "any": [
            "flights from", "flight from", "routes from", "route from",
            "flights between", "routes between", "fly from", "flying from",
            "departing from", "going from", "travel from", "flights to",
            "route between", "travel between"
        ],
        "exclude": ["delay", "late", "early", "punctual", "on-time",
                   "frequent", "busiest", "most", "rank", "top"]
    },

    # 2. route_delay_analysis
    {
        "intent": "route_delay_analysis",
        "any": [
            "delay", "delays", "late", "early", "punctual", "on-time",
            "arrival time", "delayed", "punctuality", "late arrival",
            "early arrival", "on time"
        ],
        "exclude": ["airport", "airports"]
    },

    # 3. food_satisfaction_analysis
    {
        "intent": "food_satisfaction_analysis",
        "any": [
            "food", "meal", "meals", "catering", "dining", "cuisine",
            "menu", "food score", "meal quality", "food satisfaction",
            "food rating"
        ],
        "exclude": []
    },

    # 4. loyalty_miles_analysis
    {
        "intent": "loyalty_miles_analysis",
        "any": [
            "loyalty", "premier", "elite", "member", "membership tier",
            "loyalty program", "loyalty level", "miles by loyalty",
            "distance by loyalty", "premier 1k", "gold member", "silver member"
        ],
        "exclude": []
    },

    # 5. route_frequency_ranking
    {
        "intent": "route_frequency_ranking",
        "any": [ "busiest", "popular", "frequent", "most", "rank", "top",
            "most frequent route", "busiest route", "popular route",
            "top route", "most common route", "highest traffic route",
            "route frequency", "route traffic", "route volume",
            "how many flights", "number of flights", "frequent routes",
            "most traveled route", "route rank"
        ],
        "exclude": ["airport"]
    },

    # 6. airport_traffic_analysis
    {
        "intent": "airport_traffic_analysis",
        "any": [
            "busiest airport", "most visited airport", "popular airport",
            "top airport", "airport traffic", "airport volume",
            "airport frequency", "hub airport", "major airport",
            "most departures", "most arrivals", "busiest airports",
            "airport rank", "hub"
        ],
        "exclude": ["route", "from.*to"]
    },

    # 7. fleet_performance_analysis
    {
        "intent": "fleet_performance_analysis",
        "any": [
            "fleet", "aircraft", "plane", "airplane", "boeing", "airbus",
            "737", "320", "a320", "b737", "fleet type", "aircraft type",
            "boeing 737", "airbus 320", "fleet performance"
        ],
        "exclude": []
    },

    # 8. passenger_segmentation_query
    {
        "intent": "passenger_segmentation_query",
        "any": [
            "generation", "millennial", "millennials", "gen z", "gen x",
            "baby boomer", "boomers", "boomer", "demographic", "age group",
            "passenger type", "customer segment", "generation z"
        ],
        "exclude": []
    },

    # 9. overall_satisfaction_analysis
    {
        "intent": "overall_satisfaction_analysis",
        "any": [
            "satisfaction", "satisfied", "happy", "unhappy",
            "pleased", "content", "experience", "overall rating",
            "passenger satisfaction", "customer satisfaction",
            "overall satisfaction", "happy passengers"
        ],
        "exclude": ["food", "meal"]
    },

    # 10. journey_complexity_analysis
    {
        "intent": "journey_complexity_analysis",
        "any": [
            "connection", "connections", "layover", "layovers",
            "stopover", "multi-leg", "direct flight", "non-stop",
            "legs", "segments", "connecting flight", "number of legs",
            "direct flights", "nonstop"
        ],
        "exclude": []
    }
]

## Intent Classification - Rule Based Method.

The first method we test out for classification is simple rule-based method using the all and any signals for keyword matching.

- We utilize the `re` library for regular expressions, and use a simple function to break down the input string into a set of unique lower-case words using regular expressions.

- Then we implement the `rule_based_intent_classification` function to give every question a score per intent rule. The highest scoring intent is classified as the user intent.

In [ ]:
import re

# Normalize text by making it all lower case and stripping it from leading and trailing white spaces.
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

# Checks if any exclusion pattern is matched for this text.
def check_exclusions(text, exclusions):
    if not exclusions:
        return 0

    score = 0
    text_normalized = normalize_text(text)

    for pattern in exclusions:
        if isinstance(pattern, str):
            if pattern in text_normalized:
                score +=1
        else:
            if re.search(pattern, text_normalized):
                score +=1

    return score

# Adds a score of 1 to the total score of this intent (for this particular text), for each matched "any" keyword.
def calculate_intent_score(query, rule):
    text = normalize_text(query)

    # Check exclusions first - if any exclusion matches, score is 0
    if check_exclusions(text, rule.get("exclude", [])) > 3:
        return 0

    # Count how many keywords from 'any' list match
    score = 0
    matched_keywords = []

    for keyword in rule.get("any", []):
        if keyword in text:
            score += 1
            matched_keywords.append(keyword)

    return score, matched_keywords


# Classifies intent, by calculating score for each intent and choosing the highest scoring intent.
def classify_intent(query):
    all_scores = {}
    best_intent = None
    best_score = 0
    best_keywords = []

    for rule in INTENT_RULES:
        score, matched_keywords = calculate_intent_score(query, rule)
        intent_name = rule["intent"]
        all_scores[intent_name] = score

        if score > best_score:
            best_score = score
            best_intent = intent_name
            best_keywords = matched_keywords

    # If no matches found
    if best_score == 0:
       return "unkown"

    return best_intent

In [ ]:
test_queries = [
        # route_operations_query
        "What flights go from ORD to LAX?",
        "Show me routes between JFK and MIA",

        # route_delay_analysis
        "What's the average delay from LAX to JFK?",
        "Show me routes with delays over 60 minutes",

        # route_frequency_ranking
        "What are the top 10 most frequent routes?",
        "Which route has the most flights?",

        # airport_traffic_analysis
        "Which airports have the most departures?",
        "What are the top 5 busiest airports?",

        # food_satisfaction_analysis
        "What's the average food score for business class?",
        "How do passengers rate our meal service?",

        # overall_satisfaction_analysis
        "How many passengers were satisfied overall?",
        "What's the satisfaction rate for economy passengers?",

        # loyalty_miles_analysis
        "How many miles do premier 1K members fly?",
        "What's the total distance flown by gold members?",

        # fleet_performance_analysis
        "Which fleet has the best on-time performance?",
        "How many Boeing 737 flights do we operate?",

        # passenger_segmentation_query
        "How do millennials rate our service?",
        "What do Gen Z passengers prefer?",

        # Ambiguous cases
        "Show me the busiest routes",
        "What's the satisfaction for food?",
        "Delays at ORD",
    ]




for q in test_queries:
    print(q, "->", classify_intent(q))


What flights go from ORD to LAX? -> unkown
Show me routes between JFK and MIA -> route_operations_query
What's the average delay from LAX to JFK? -> route_delay_analysis
Show me routes with delays over 60 minutes -> route_delay_analysis
What are the top 10 most frequent routes? -> route_frequency_ranking
Which route has the most flights? -> route_frequency_ranking
Which airports have the most departures? -> route_frequency_ranking
What are the top 5 busiest airports? -> route_frequency_ranking
What's the average food score for business class? -> food_satisfaction_analysis
How do passengers rate our meal service? -> food_satisfaction_analysis
How many passengers were satisfied overall? -> overall_satisfaction_analysis
What's the satisfaction rate for economy passengers? -> overall_satisfaction_analysis
How many miles do premier 1K members fly? -> loyalty_miles_analysis
What's the total distance flown by gold members? -> loyalty_miles_analysis
Which fleet has the best on-time performance

As clear, some intents were perfectly classified.

However, other intents such as route_operations_query were sometimes mislabeled as unkown intent, airport_traffic_analysis was misclassified as route_traffic_analysis and so on.

This calls for another implementation, possibly utilizing an LLM to classify user question intents.

## Intent Classification - LLM-Based Method.

Due to the misconfigured intents, we will test another classification method using Large Language Models.

For large language models to classify correctly, we need to clearly define the intents rather than use keywords.

We will use Openrouter for LLMs, due to its simplicity, and the availability of a large set of free models.

In [ ]:
INTENT_PROMPT_TEMPLATE = """You are an intent classification assistant for an airline analytics system that helps analyze flight data, passenger satisfaction, and operational metrics.

Your task is to classify the user's query into exactly ONE of the following intents. Read each intent definition carefully and select the most appropriate match.

===== INTENT DEFINITIONS =====

1. route_operations_query
   PURPOSE: Find specific flights, routes, or operational details between two airports.
   USE WHEN: User wants to identify, list, or check flights connecting a specific origin to a specific destination.
   REQUIRES: Both origin and destination airports (e.g., "from LAX to JFK")
   EXAMPLES:
   - "What flights go from ORD to LAX?"
   - "Show me all routes between JFK and MIA"
   - "List Boeing 737 flights departing from SFO to DEN"
   - "How many economy class flights fly from ATL to BOS?"
   DO NOT USE FOR: Delay analysis, frequency rankings, or airport-wide statistics

2. route_delay_analysis
   PURPOSE: Analyze arrival delays and punctuality for specific routes.
   USE WHEN: User asks about delays, late/early arrivals, or on-time performance for flights between two specific airports.
   REQUIRES: Delay-related keywords (delay, late, early, punctual, on-time) AND route information
   EXAMPLES:
   - "What's the average delay from LAX to JFK?"
   - "Show me routes with delays over 60 minutes"
   - "Are flights between ORD and DFW usually on time?"
   - "Which routes have the worst punctuality?"
   DO NOT USE FOR: Airport-level delay statistics or counting flights

3. route_frequency_ranking
   PURPOSE: Rank routes by volume and identify most/least frequent routes.
   USE WHEN: User asks about which routes are most common, popular, or have the highest number of flights.
   FOCUSES ON: Route-level traffic comparison across the dataset
   EXAMPLES:
   - "What are the top 10 most frequent routes?"
   - "Which route has the most flights?"
   - "Show me the least traveled routes"
   - "Rank routes by number of journeys"
   DO NOT USE FOR: Airport-level traffic or specific route lookups

4. airport_traffic_analysis
   PURPOSE: Analyze airport-level traffic, hub importance, and airport rankings.
   USE WHEN: User asks about busiest airports, airport traffic volume, or which airports handle the most flights.
   FOCUSES ON: Airport-centric analysis (departures, arrivals, total traffic)
   EXAMPLES:
   - "Which airports have the most departures?"
   - "What are the top 5 busiest airports?"
   - "Show me airports ranked by total traffic"
   - "Which airport is the biggest hub?"
   DO NOT USE FOR: Route-specific queries or passenger satisfaction

5. food_satisfaction_analysis
   PURPOSE: Analyze in-flight food and meal service quality.
   USE WHEN: User asks specifically about food, meals, catering, or dining satisfaction.
   FOCUSES ON: Food satisfaction scores and meal quality metrics
   EXAMPLES:
   - "What's the average food score for business class?"
   - "How do passengers rate our meal service?"
   - "Show me food satisfaction by route"
   - "Which flights have the best catering reviews?"
   DO NOT USE FOR: Overall satisfaction (use overall_satisfaction_analysis instead)

6. overall_satisfaction_analysis
   PURPOSE: Compute composite passenger satisfaction using multiple factors.
   USE WHEN: User asks about overall satisfaction, happy passengers, or general experience quality.
   CONSIDERS: Multiple factors including food scores, delays, journey complexity, and distance
   EXAMPLES:
   - "How many passengers were satisfied overall?"
   - "What's the satisfaction rate for economy passengers?"
   - "Show me overall customer satisfaction trends"
   - "Which generation is most satisfied with our service?"
   DO NOT USE FOR: Food-only satisfaction (use food_satisfaction_analysis)

7. loyalty_miles_analysis
   PURPOSE: Analyze flight distance and mileage patterns by loyalty program tier.
   USE WHEN: User asks about miles flown, distance traveled, or mileage segmented by loyalty level.
   REQUIRES: Keywords about loyalty (premier, elite, member, loyalty level) AND mileage/distance
   EXAMPLES:
   - "How many miles do premier 1K members fly on average?"
   - "What's the total distance flown by gold members?"
   - "Show me average miles by loyalty level"
   - "Which loyalty tier travels the most?"
   DO NOT USE FOR: Satisfaction analysis or route frequency

8. fleet_performance_analysis
   PURPOSE: Analyze aircraft performance, utilization, and metrics by fleet type.
   USE WHEN: User asks about specific aircraft types, planes, or fleet comparison.
   REQUIRES: Keywords about aircraft (Boeing, Airbus, 737, A320, fleet, aircraft, plane)
   EXAMPLES:
   - "Which fleet has the best on-time performance?"
   - "How many Boeing 737 flights do we operate?"
   - "Show me satisfaction scores by aircraft type"
   - "What's the average delay for Airbus A320 flights?"
   DO NOT USE FOR: Route-specific queries without fleet context

9. passenger_segmentation_query
   PURPOSE: Analyze passenger demographics, behavior, and preferences by segment.
   USE WHEN: User asks about passenger types, generations, age groups, or demographic patterns.
   REQUIRES: Keywords about demographics (millennial, Gen Z, Gen X, baby boomer, generation)
   EXAMPLES:
   - "How do millennials rate our service?"
   - "What class do Gen Z passengers prefer?"
   - "Show me travel patterns for baby boomers"
   - "Which generation has the highest loyalty participation?"
   DO NOT USE FOR: General satisfaction without demographic context

10. journey_complexity_analysis
    PURPOSE: Analyze trip complexity, connections, and multi-leg journey patterns.
    USE WHEN: User asks about connections, layovers, direct flights, or journey segments.
    REQUIRES: Keywords about connections (layover, stopover, multi-leg, direct, non-stop, segments)
    EXAMPLES:
    - "How many direct flights do we offer?"
    - "What's the average number of connections for LAX to JFK?"
    - "Show me satisfaction for multi-leg journeys"
    - "Which routes require the most layovers?"
    DO NOT USE FOR: Simple route lookups without complexity focus

11. unknown
    PURPOSE: Catch-all for queries that don't match any intent.
    USE WHEN: The query is unclear, too vague, or doesn't relate to airline analytics.
    EXAMPLES:
    - "Hello, how are you?"
    - "What's the weather like?"
    - "Tell me a joke"
    - Gibberish or off-topic queries

===== CLASSIFICATION GUIDELINES =====

- If the query mentions TWO AIRPORTS with "from...to" or "between", consider route-specific intents (1, 2, or 3)
- If the query mentions "delay", "late", or "early" with a route, choose intent 2 (route_delay_analysis)
- If the query mentions "food" or "meal", choose intent 5 (food_satisfaction_analysis), NOT 6
- If the query mentions "satisfaction" without "food", choose intent 6 (overall_satisfaction_analysis)
- If the query mentions "busiest airports" or "most arrivals/departures", choose intent 4 (airport_traffic_analysis)
- If the query mentions "busiest routes" or "most frequent routes", choose intent 3 (route_frequency_ranking)
- If the query mentions loyalty levels AND miles/distance, choose intent 7 (loyalty_miles_analysis)
- If the query mentions aircraft/fleet types, choose intent 8 (fleet_performance_analysis)
- If the query mentions generations or demographics, choose intent 9 (passenger_segmentation_query)
- If the query mentions connections or layovers, choose intent 10 (journey_complexity_analysis)
- When in doubt between two intents, choose the MORE SPECIFIC one

===== YOUR TASK =====

User query: "{user_query}"

Analyze the query above and return ONLY the intent name from the list below. Do not include explanations, reasoning, or additional text.

Valid intents:
- route_operations_query
- route_delay_analysis
- route_frequency_ranking
- airport_traffic_analysis
- food_satisfaction_analysis
- overall_satisfaction_analysis
- loyalty_miles_analysis
- fleet_performance_analysis
- passenger_segmentation_query
- journey_complexity_analysis
- unknown

Your response (intent name only):"""

In [ ]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

def classify_intent_llm(user_query):
    prompt = INTENT_PROMPT_TEMPLATE.format(user_query=user_query)

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        data=json.dumps({
            "model": "kwaipilot/kat-coder-pro:free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.0
        })
    )

    response.raise_for_status()
    result = response.json()
    intent = result["choices"][0]["message"]["content"].strip()
    return intent

In [ ]:
# test_queries = [
#         # route_operations_query
#         "What flights go from ORD to LAX?",
#         "Show me routes between JFK and MIA",

#         # route_delay_analysis
#         "What's the average delay from LAX to JFK?",
#         "Show me routes with delays over 60 minutes",

#         # route_frequency_ranking
#         "What are the top 10 most frequent routes?",
#         "Which route has the most flights?",

#         # airport_traffic_analysis
#         "Which airports have the most departures?",
#         "What are the top 5 busiest airports?",

#         # food_satisfaction_analysis
#         "What's the average food score for business class?",
#         "How do passengers rate our meal service?",

#         # overall_satisfaction_analysis
#         "How many passengers were satisfied overall?",
#         "What's the satisfaction rate for economy passengers?",

#         # loyalty_miles_analysis
#         "How many miles do premier 1K members fly?",
#         "What's the total distance flown by gold members?",

#         # fleet_performance_analysis
#         "Which fleet has the best on-time performance?",
#         "How many Boeing 737 flights do we operate?",

#         # passenger_segmentation_query
#         "How do millennials rate our service?",
#         "What do Gen Z passengers prefer?",

#         # Ambiguous cases
#         "Show me the busiest routes",
#         "What's the satisfaction for food?",
#         "Delays at ORD",
#     ]




# for q in test_queries:
#     print(q, "->", classify_intent_llm(q))


 By comparing both tests, we can see that the LLM classifier is much better, and it didn't misclassify any intent making it a much stronger candidate than the rule-based matching attempted in the first trial.

Therefore, we will go forward with using the LLM-based classifier.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Named Entity Recognition

- spaCy’s pretrained NER can detect: locations, organizations, numbers, dates and so on. However, it wouldn't know data such as airport codes like ORD, LAX, flight numbers, loyalty levels and so on.

- So, we will use spaCy and some rule-based extraction for the knowledge graph entities/attributes...


The existing knowledge graph schema:

1. ​Nodes:
- **Passenger**: record_locator (unique identifier), loyalty_program_level, generation
- **Journey**: feedback_ID (unique identifier), food_satisfaction_score,
arrival_delay_minutes , actual_flown_miles ,number_of_legs, passenger_class
- **Flight**: flight_number, fleet_type_description (uniquely identified by both
properties)
- **Airport**: station_code (e.g., ORD, LAX)

2. Relations:
- (Passenger)-[:**TOOK** ]->(Journey)
- (Journey)-[:**ON** ]->(Flight)
- (Flight)-[:**DEPARTS_FROM** ]->(Airport)
- (Flight)-[:**ARRIVES_AT** ]->(Airport)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
test_queries = [
    "Show me flights from ORD to LAX",
    "Which flights go from JFK to SFO?",
    "List all routes from LAX to ORD",
    "Are there flights from ATL to JFK?",

    "What is the busiest route from LAX airport?",
    "How many miles do elite 1000 members fly?",
    "What is the average food score for ORD to JFK flight?"
]

for q in test_queries:
    print(f"**Query** : {q}")
    doc = nlp(q)
    for ent in doc.ents:
        print(ent.text, ent.label_)


**Query** : Show me flights from ORD to LAX
ORD ORG
**Query** : Which flights go from JFK to SFO?
JFK PERSON
SFO ORG
**Query** : List all routes from LAX to ORD
ORD ORG
**Query** : Are there flights from ATL to JFK?
ATL ORG
JFK PERSON
**Query** : What is the busiest route from LAX airport?
LAX airport FAC
**Query** : How many miles do elite 1000 members fly?
1000 CARDINAL
**Query** : What is the average food score for ORD to JFK flight?
ORD ORG
JFK PERSON


It's obvious spacy on its own mislabels airport codes and loyalty levels used. For this reason, we must apply a rule-based keyword matching system.

In [ ]:
# Airport code extraction
import pandas as pd
import re
AIRPORT_CODE_PATTERN = re.compile(r"\b[A-Z]{3}\b")


def load_valid_airports(csv_path = "/content/Airline_surveys_sample.csv"):
    df = pd.read_csv(csv_path)

    origin_codes = df["origin_station_code"].astype(str).str.upper()
    dest_codes = df["destination_station_code"].astype(str).str.upper()

    valid_airports = set(origin_codes).union(set(dest_codes))

    return valid_airports


VALID_AIRPORTS = load_valid_airports()

# extracts all three letter words.
# checks against valid airports from the csv to confirm.
def extract_airports(text, valid_airports = VALID_AIRPORTS):
    candidates = AIRPORT_CODE_PATTERN.findall(text.upper())
    return [c for c in candidates if c in valid_airports]


extract_airports("What is the average food score for IAX to LAX flight?")

['IAX', 'LAX']

In [ ]:
# Route extraction for journeys.
def extract_route(text):
    airports = extract_airports(text)
    origin = destination = None

    if len(airports) >= 2:
        origin, destination = airports[:2]

    return origin, destination


print(extract_route("What is the average food score from IAX to LAX flight?"))
print(extract_route("What is the average food score in IAX to LAX flight?"))

('IAX', 'LAX')
('IAX', 'LAX')


In [ ]:
# Numeric entity extraction using spaCy (delays, miles)
def extract_numbers(doc):
    numbers = []
    for ent in doc.ents:
        if ent.label_ in ["CARDINAL", "QUANTITY", "TIME"]: # since 60 minutes showed up as TIME, and 60km showed up as QUANTITY
            nums = re.findall(r'\d+', ent.text) # extract text if the extracted entity includes "minutes" or "km" or so on.
            numbers.extend(nums)
    return numbers

doc = nlp("Journeys with delay over 60 minutes?")
extract_numbers(doc)

['60']

In [ ]:
# Returns positive value for delays/ late arrivals and negative values for early arrivals since both are used in the knowledge graph.
def extract_delay_threshold(text):
    doc = nlp(text.lower())
    text_lower = text.lower()

    # Find all numbers in the text
    for ent in doc.ents:
        if ent.label_ in {"CARDINAL", "QUANTITY", "TIME"}:
            nums = re.findall(r'\d+', ent.text)
            if nums:
                value = int(nums[0])

                # Early indicators (negative)
                if any(k in text_lower for k in ["early", "earlier", "ahead"]):
                    return -value

                # Late/delay indicators (positive)
                if any(k in text_lower for k in ["delay", "late", "later", "behind"]):
                    return value

                # Default: if time-related words present, assume delay
                if any(k in text_lower for k in ["time", "minute", "hour"]):
                    return value

    return None

print(extract_delay_threshold("Journeys with delay over 60 minutes?"))
print(extract_delay_threshold("Journeys that were at least 10 minutes early?"))
print(extract_delay_threshold("Journeys that were late at maximum 24 hours?"))
print(extract_delay_threshold("Show me trips delayed by 15 minutes"))
print(extract_delay_threshold("Arrivals 5 minutes ahead of schedule"))

60
-10
24
15
-5


Now, we need to also extract other categorical features such as Loyalty Level and Passenger Class.

These need to be validated against the csv, similar to how we handled airports.

In [ ]:
def load_valid_categories(csv_path = "/content/Airline_surveys_sample.csv"):
    df = pd.read_csv(csv_path)

    valid_passenger_classes = df["passenger_class"].tolist()


    valid_loyalty_levels = df["loyalty_program_level"].tolist()

    return {
        "passenger_class": set(valid_passenger_classes),
        "loyalty_level": set(valid_loyalty_levels)
    }


VALID_CLASSES = load_valid_categories()["passenger_class"]
def extract_passenger_class(text, valid_classes = VALID_CLASSES):
    text = text.lower()

    for cls in valid_classes:
        if cls.lower() in text:
            return cls.capitalize()

    return None



VALID_LEVELS = load_valid_categories()["loyalty_level"]
print(VALID_LEVELS)
def extract_loyalty_level(text, valid_levels = VALID_LEVELS):
    text = text.lower()

    for lvl in valid_levels:
        if lvl.lower() in text:
            return lvl.lower()

    return None


print(extract_passenger_class("What is the highest rated economy journey?"))
print(extract_loyalty_level("What is the most frequent premier 1k trip?"))
print(extract_loyalty_level("What is the most frequent trip for NBK?"))

{'NBK', 'global services', 'premier gold', 'premier platinum', 'premier 1k', 'non-elite', 'premier silver'}
Economy
premier 1k
nbk


In [ ]:
#Extract valid fleet types
def load_valid_fleet_types(csv_path="/content/Airline_surveys_sample.csv"):
    df = pd.read_csv(csv_path)
    return set(df["fleet_type_description"].dropna())

VALID_FLEET_TYPES = load_valid_fleet_types()

def extract_fleet_type(text, valid_fleet_types=VALID_FLEET_TYPES):
    text_lower = text.lower()
    for fleet in valid_fleet_types:
        if fleet.lower() in text_lower:
            return fleet
    return None

In [ ]:
#Extract generations
def load_valid_generations(csv_path="/content/Airline_surveys_sample.csv"):
    df = pd.read_csv(csv_path)
    return set(df["generation"].dropna())

VALID_GENERATIONS = load_valid_generations()

def extract_generation(text, valid_generations=VALID_GENERATIONS):
    text_lower = text.lower()
    for gen in valid_generations:
        if gen.lower() in text_lower:
            return gen
    return None

In [ ]:
def extract_miles_threshold(text):
    doc = nlp(text.lower())

    for ent in doc.ents:
        if ent.label_ in {"CARDINAL", "QUANTITY"}:
            nums = re.findall(r'\d+', ent.text)
            if nums and any(k in text.lower() for k in ["mile", "km", "distance", "flown"]):
                return int(nums[0])
    return None

In [ ]:
def extract_number_of_legs(text):
    doc = nlp(text.lower())

    for ent in doc.ents:
        if ent.label_ in {"CARDINAL"}:
            nums = re.findall(r'\d+', ent.text)
            if nums and any(k in text.lower() for k in ["leg", "segment", "stop", "connection", "layover"]):
                return int(nums[0])

    # Handle special cases
    if "direct" in text.lower() or "nonstop" in text.lower() or "non-stop" in text.lower():
        return 1

    return None

In [ ]:
def extract_satisfaction_threshold(text):
    """Extract satisfaction score thresholds"""
    doc = nlp(text.lower())

    for ent in doc.ents:
        if ent.label_ in {"CARDINAL"}:
            nums = re.findall(r'\d+', ent.text)
            if nums and any(k in text.lower() for k in ["satisfaction", "score", "rating", "rated", "above", "below", "over", "under"]):
                return int(nums[0])
    return None

In [ ]:
# Extract top n for ranking queries
def extract_top_n(text):
    doc = nlp(text.lower())
    text_lower = text.lower()

    # Look for patterns like "top N", "first N", "best N", "worst N"
    ranking_keywords = ["top", "first", "best", "worst", "bottom", "least", "most", "rank"]

    for ent in doc.ents:
        if ent.label_ in {"CARDINAL"}:
            nums = re.findall(r'\d+', ent.text)
            if nums:
                # Check if there's a ranking keyword near this number
                for keyword in ranking_keywords:
                    # Check patterns like "top 5" or "5 best"
                    if f"{keyword} {nums[0]}" in text_lower or f"{nums[0]} {keyword}" in text_lower:
                        return int(nums[0])

    # Default to 10 if keywords present but no number
    if any(k in text_lower for k in ranking_keywords):
        return 10  # Reasonable default

    return None

## Named Entity Extraction Function

Uses all the previous functions to build a catalog of named entities to use for querying.

In [ ]:
def extract_all_entities(text):
    # Extract route information
    origin, destination = extract_route(text)

    # Run all extractors
    entities = {
        "origin_airport": origin,
        "destination_airport": destination,
        "all_airports": extract_airports(text),
        "delay_threshold": extract_delay_threshold(text),
        "miles_threshold": extract_miles_threshold(text),
        "satisfaction_threshold": extract_satisfaction_threshold(text),
        "number_of_legs": extract_number_of_legs(text),
        "top_n": extract_top_n(text),
        "passenger_class": extract_passenger_class(text),
        "loyalty_level": extract_loyalty_level(text),
        "fleet_type": extract_fleet_type(text),
        "generation": extract_generation(text)
    }

    return entities

In [ ]:
extraction_test_queries = [
        "What flights go from IAX to LAX?",

        "Show me routes with delays over 60 minutes from IAX to EWX",

        "What are the top 10 most frequent routes?",

        "Rank the 5 busiest airports",

        "What's the average food score above 8 for economy class?",

        "Show me the top 20 satisfied premier 1K members",

        "How many Premier Gold members fly over 5000 miles?",

        "Show me B777-200 flights with delays",

        "How do millennials rate economy class?",

        "Show me direct flights from LAX to EWX",

        "What are the top 5 routes for Gen Z passengers in economy class with food scores above 7?"
    ]

def print_extracted_entities(text, entities):
    print(f"\n{'='*80}")
    print(f"QUERY: {text}")

    for key, value in entities.items():
        if value:
            print(f"  • {key}: {value}")

    print(f"{'='*80}\n")



for query in extraction_test_queries:
        entities = extract_all_entities(query)
        print_extracted_entities(query, entities)


QUERY: What flights go from IAX to LAX?
  • origin_airport: IAX
  • destination_airport: LAX
  • all_airports: ['IAX', 'LAX']


QUERY: Show me routes with delays over 60 minutes from IAX to EWX
  • origin_airport: IAX
  • destination_airport: EWX
  • all_airports: ['IAX', 'EWX']
  • delay_threshold: 60


QUERY: What are the top 10 most frequent routes?
  • top_n: 10


QUERY: Rank the 5 busiest airports
  • top_n: 10


QUERY: What's the average food score above 8 for economy class?
  • satisfaction_threshold: 8
  • passenger_class: Economy


QUERY: Show me the top 20 satisfied premier 1K members
  • top_n: 20
  • loyalty_level: premier 1k


QUERY: How many Premier Gold members fly over 5000 miles?
  • miles_threshold: 5000
  • loyalty_level: premier gold


QUERY: Show me B777-200 flights with delays
  • fleet_type: B777-200


QUERY: How do millennials rate economy class?
  • passenger_class: Economy
  • generation: Millennial


QUERY: Show me direct flights from LAX to EWX
  • origin_a

# Graph Retrieval Layer
## Baseline
Use deterministic cypher queries to retrieve relevant information based on intent classification and extracted entities.

In [ ]:
class Neo4jConnection: # Class responsible for neo4j operations.

    # constructor calls driver to create driver
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))


    def close(self):
        if self.driver:
            self.driver.close()


    # executes a cypher query and returns the results.
    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters or {})
            return [record.data() for record in result]

In [ ]:
# ==================== CYPHER QUERY TEMPLATES ====================

CYPHER_TEMPLATES = {

    # Intent 1: Find flights between two airports
    "route_operations_query": """
        MATCH (origin:Airport {station_code: $origin})<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport {station_code: $destination})
        MATCH (j:Journey)-[:ON]->(f)
        RETURN f.flight_number as flight_number,
               f.fleet_type_description as fleet_type,
               COUNT(j) as journey_count,
               origin.station_code as origin,
               dest.station_code as destination
        ORDER BY journey_count DESC
        LIMIT $top_n
    """,

    # Intent 2: Analyze delays for routes (can be specific route or all routes)
    "route_delay_analysis": """
        MATCH (origin:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport)
        MATCH (j:Journey)-[:ON]->(f)
        WHERE j.arrival_delay_minutes IS NOT NULL
          AND ($origin IS NULL OR origin.station_code = $origin)
          AND ($destination IS NULL OR dest.station_code = $destination)
          AND ($delay_threshold IS NULL OR j.arrival_delay_minutes > $delay_threshold)
        WITH f, origin, dest,
             COUNT(j) as journey_count,
             AVG(j.arrival_delay_minutes) as avg_delay,
             MIN(j.arrival_delay_minutes) as min_delay,
             MAX(j.arrival_delay_minutes) as max_delay,
             SUM(CASE WHEN j.arrival_delay_minutes > 0 THEN 1 ELSE 0 END) as delayed_count,
             SUM(CASE WHEN j.arrival_delay_minutes <= 0 THEN 1 ELSE 0 END) as ontime_count
        RETURN f.flight_number as flight_number,
               f.fleet_type_description as fleet_type,
               origin.station_code as origin,
               dest.station_code as destination,
               journey_count,
               avg_delay,
               min_delay,
               max_delay,
               delayed_count,
               ontime_count,
               ROUND(100.0 * ontime_count / journey_count, 1) as ontime_rate
        ORDER BY avg_delay DESC
        LIMIT $top_n
    """,

    # Intent 3: Rank routes by frequency
    "route_frequency_ranking": """
        MATCH (origin:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport)
        MATCH (j:Journey)-[:ON]->(f)
        WHERE origin.station_code <> dest.station_code
        WITH origin.station_code as origin,
             dest.station_code as destination,
             COUNT(j) as journey_count
        ORDER BY journey_count DESC
        LIMIT $top_n
        RETURN origin, destination, journey_count
    """,

    # Intent 4: Analyze airport traffic volume
    "airport_traffic_analysis": """
        MATCH (a:Airport)
        OPTIONAL MATCH (a)<-[:DEPARTS_FROM]-(f1:Flight)<-[:ON]-(j1:Journey)
        OPTIONAL MATCH (a)<-[:ARRIVES_AT]-(f2:Flight)<-[:ON]-(j2:Journey)
        WITH a.station_code as airport_code,
             COUNT(DISTINCT j1) as departures,
             COUNT(DISTINCT j2) as arrivals,
             COUNT(DISTINCT j1) + COUNT(DISTINCT j2) as total_traffic
        WHERE total_traffic > 0
        ORDER BY total_traffic DESC
        LIMIT $top_n
        RETURN airport_code, departures, arrivals, total_traffic
    """,

    # Intent 5: Analyze food satisfaction (shows journeys matching criteria)
    "food_satisfaction_analysis": """
       MATCH (p:Passenger)-[:TOOK]->(j:Journey)-[:ON]->(f:Flight)
MATCH (f)-[:DEPARTS_FROM]->(origin:Airport)
MATCH (f)-[:ARRIVES_AT]->(dest:Airport)
WHERE j.food_satisfaction_score IS NOT NULL
  AND ($passenger_class IS NULL OR j.passenger_class = $passenger_class)
  AND ($satisfaction_threshold IS NULL OR j.food_satisfaction_score >= $satisfaction_threshold)
  AND ($loyalty_level IS NULL OR p.loyalty_program_level = $loyalty_level)
  AND ($generation IS NULL OR p.generation = $generation)
  AND ($origin IS NULL OR origin.station_code = $origin)
  AND ($destination IS NULL OR dest.station_code = $destination)

WITH
  j, p, origin, dest,
  CASE
    WHEN $origin IS NOT NULL AND $destination IS NOT NULL THEN origin.station_code + ' → ' + dest.station_code
    WHEN $passenger_class IS NOT NULL THEN j.passenger_class
    WHEN $generation IS NOT NULL THEN p.generation
    WHEN $loyalty_level IS NOT NULL THEN p.loyalty_program_level
    ELSE 'Overall'
  END AS group_by

WITH group_by,
     COUNT(j) AS total_journeys,
     AVG(j.food_satisfaction_score) AS avg_food_score,
     MIN(j.food_satisfaction_score) AS min_score,
     MAX(j.food_satisfaction_score) AS max_score,
     SUM(CASE WHEN j.food_satisfaction_score >= 3 THEN 1 ELSE 0 END) AS high_satisfaction_count

RETURN group_by,
       total_journeys,
       avg_food_score,
       min_score,
       max_score,
       high_satisfaction_count,
       ROUND(100.0 * high_satisfaction_count / total_journeys, 1) AS satisfaction_rate
ORDER BY avg_food_score DESC
LIMIT $top_n

    """,

    # Intent 6: Compute overall passenger satisfaction using weighted formula
   "overall_satisfaction_analysis": """
        MATCH (p:Passenger)-[:TOOK]->(j:Journey)
        WHERE j.food_satisfaction_score IS NOT NULL
          AND j.arrival_delay_minutes IS NOT NULL
          AND j.number_of_legs IS NOT NULL
          AND j.actual_flown_miles IS NOT NULL
        WITH p, j,
             CASE WHEN abs(j.arrival_delay_minutes)/20.0 > 5 THEN 5 ELSE abs(j.arrival_delay_minutes)/20.0 END AS delay_raw,
             CASE WHEN j.number_of_legs * 1.5 > 5 THEN 5 ELSE j.number_of_legs * 1.5 END AS legs_raw,
             CASE WHEN j.actual_flown_miles / 3000.0 > 5 THEN 5 ELSE j.actual_flown_miles / 3000.0 END AS miles_raw
        WITH p, j,
             5 - delay_raw AS delay_score,
             5 - (CASE WHEN j.number_of_legs * 1.5 > 5 THEN 5 ELSE j.number_of_legs * 1.5 END) AS legs_score,
             5 - (CASE WHEN j.actual_flown_miles / 3000.0 > 5 THEN 5 ELSE j.actual_flown_miles / 3000.0 END) AS miles_score
        WITH p, j,
             ROUND(0.5 * j.food_satisfaction_score +
                   0.35 * delay_score +
                   0.1 * legs_score +
                   0.05 * miles_score, 1) AS overall_satisfaction_score
        WHERE overall_satisfaction_score > 3
        WITH COUNT(DISTINCT p) as satisfied_passengers,
             AVG(overall_satisfaction_score) as avg_satisfaction,
             MIN(overall_satisfaction_score) as min_satisfaction,
             MAX(overall_satisfaction_score) as max_satisfaction
        RETURN satisfied_passengers,
               avg_satisfaction,
               min_satisfaction,
               max_satisfaction
    """,

    # Intent 7: Analyze miles by loyalty level
"loyalty_miles_analysis": """
        MATCH (p:Passenger)-[:TOOK]->(j:Journey)
        WHERE j.actual_flown_miles IS NOT NULL
          AND p.loyalty_program_level IS NOT NULL
        WITH p.loyalty_program_level as loyalty_level,
             SUM(j.actual_flown_miles) as total_miles,
             AVG(j.actual_flown_miles) as avg_miles,
             COUNT(j) as journey_count
        ORDER BY total_miles DESC
        RETURN loyalty_level, total_miles, avg_miles, journey_count
    """,

    # Intent 8: Analyze fleet performance
    "fleet_performance_analysis": """
        MATCH (f:Flight)
        MATCH (j:Journey)-[:ON]->(f)
        WHERE f.fleet_type_description IS NOT NULL
          AND j.arrival_delay_minutes IS NOT NULL
          AND ($fleet_type IS NULL OR f.fleet_type_description = $fleet_type)
        WITH f.fleet_type_description as fleet_type,
             COUNT(j) as total_flights,
             AVG(j.arrival_delay_minutes) as avg_delay,
             AVG(j.food_satisfaction_score) as avg_food_score,
             SUM(CASE WHEN j.arrival_delay_minutes <= 15 THEN 1 ELSE 0 END) as ontime_count
        RETURN fleet_type,
               total_flights,
               avg_delay,
               avg_food_score,
               ROUND(100.0 * ontime_count / total_flights, 2) as ontime_rate
        ORDER BY total_flights DESC
        LIMIT $top_n
    """,

    # Intent 9: Analyze passenger demographics
    "passenger_segmentation_query": """
        MATCH (p:Passenger)-[:TOOK]->(j:Journey)
        WHERE p.generation IS NOT NULL
          AND ($generation IS NULL OR p.generation = $generation)
          AND ($passenger_class IS NULL OR j.passenger_class = $passenger_class)
          AND ($satisfaction_threshold IS NULL OR j.food_satisfaction_score >= $satisfaction_threshold)
        WITH p.generation as generation,
             COUNT(DISTINCT p) as passenger_count,
             COUNT(j) as journey_count,
             AVG(j.food_satisfaction_score) as avg_food_satisfaction,
             AVG(j.actual_flown_miles) as avg_miles
        ORDER BY passenger_count DESC
        RETURN generation, passenger_count, journey_count, avg_food_satisfaction, avg_miles
    """,

    # Intent 10: Analyze journey complexity (connections/legs)
    "journey_complexity_analysis": """
        MATCH (j:Journey)-[:ON]->(f:Flight)
        WHERE j.number_of_legs IS NOT NULL
        WITH j.number_of_legs as legs,
             COUNT(j) as journey_count,
             AVG(j.food_satisfaction_score) as avg_satisfaction,
             AVG(j.arrival_delay_minutes) as avg_delay
        ORDER BY legs
        RETURN legs, journey_count, avg_satisfaction, avg_delay
    """
}

In [ ]:
def build_parameters(entities):
    params = {}

    # Route parameters (None means no filter applied)
    params["origin"] = entities.get("origin_airport")
    params["destination"] = entities.get("destination_airport")

    # Thresholds
    params["top_n"] = entities.get("top_n") or 10  # Default to 10

    params["delay_threshold"] = entities.get("delay_threshold")
    params["miles_threshold"] = entities.get("miles_threshold")
    params["satisfaction_threshold"] = entities.get("satisfaction_threshold")
    params["number_of_legs"] = entities.get("number_of_legs")

    # Categorical features (None means no filter applied)
    params["passenger_class"] = entities.get("passenger_class")
    params["loyalty_level"] = entities.get("loyalty_level")
    params["fleet_type"] = entities.get("fleet_type")
    params["generation"] = entities.get("generation")

    return params

def validate_required_entities(intent, entities):
    required_entities_map = {
        "route_operations_query": ["origin_airport", "destination_airport"],
        "route_delay_analysis": [],
        "route_frequency_ranking": [],
        "airport_traffic_analysis": [],
        "food_satisfaction_analysis": [],
        "overall_satisfaction_analysis": [],
        "loyalty_miles_analysis": [],
        "fleet_performance_analysis": [],
        "passenger_segmentation_query": [],
        "journey_complexity_analysis": []
    }

    required = required_entities_map.get(intent, [])
    missing = [ent for ent in required if not entities.get(ent)]

    return len(missing) == 0, missing

In [ ]:
def query_knowledge_graph(user_query, intent, entities, neo4j_conn):
    # Check if intent is unknown
    if intent == "unknown":
        return {
            "query": user_query,
            "intent": intent,
            "entities": entities,
            "results": None,
            "error": "Could not understand the query. Please rephrase."
        }

    # Validate required entities
    is_valid, missing = validate_required_entities(intent, entities)
    if not is_valid:
        return {
            "query": user_query,
            "intent": intent,
            "entities": entities,
            "results": None,
            "error": f"Missing required entities: {', '.join(missing)}"
        }

    # Get the appropriate Cypher template
    cypher_query = CYPHER_TEMPLATES.get(intent)
    if not cypher_query:
        return {
            "query": user_query,
            "intent": intent,
            "entities": entities,
            "results": None,
            "error": f"No query template found for intent: {intent}"
        }

    # Build parameters from entities
    parameters = build_parameters(entities)

    # Execute query
    try:
        results = neo4j_conn.execute_query(cypher_query, parameters)

        return {
            "query": user_query,
            "intent": intent,
            "entities": entities,
            "parameters": parameters,
            "results": results,
            "result_count": len(results)
        }

    except Exception as e:
        return {
            "query": user_query,
            "intent": intent,
            "entities": entities,
            "parameters": parameters,
            "results": None,
            "error": f"Query execution failed: {str(e)}"
        }

In [ ]:
def process_user_query(user_query, neo4j_conn, classification_mode='llm'):

    # Step 1: Classify intent
    if classification_mode == 'llm':
      intent_result = classify_intent_llm(user_query)
    else:
      intent_result = classify_intent(user_query)

    # Step 2: Extract entities
    entities = extract_all_entities(user_query)

    # Step 3: Query knowledge graph
    response = query_knowledge_graph(user_query, intent_result, entities, neo4j_conn)

    return response

A pretty-print function to be able to interpret results as we still don't have a GUI.

In [ ]:
def format_results(response):
    """Pretty print query results in a readable format"""

    def safe_float(value, default=0.0):
        try:
            return float(value)
        except (TypeError, ValueError):
            return default

    def safe_int(value, default=0):
        try:
            return int(value)
        except (TypeError, ValueError):
            return default

    # Handle Neo4j EagerResult objects
    results = response.get('results')
    if results and hasattr(results, 'records'):
        results = [dict(record) for record in results.records]
    elif not results:
        results = []

    print("\n" + "=" * 100)
    print(f"📝 QUERY: {response.get('query', 'N/A')}")
    print("=" * 100)

    # Show intent and confidence
    print(f"\n🎯 Intent: {response.get('intent', 'N/A')}")
    if response.get('intent_confidence'):
        print(
            f"   Confidence: {response.get('intent_confidence')} "
            f"(score: {response.get('intent_score', 0)})"
        )

    # Show extracted entities
    entities = {
        k: v for k, v in response.get('entities', {}).items()
        if v not in (None, [], '')
    }
    if entities:
        print("\n🔍 Extracted Entities:")
        for key, value in entities.items():
            print(f"   • {key}: {value}")

    # Show error if present
    if response.get('error'):
        print(f"\n❌ ERROR: {response['error']}")
        print("\n" + "=" * 100 + "\n")
        return

    # Show results
    result_count = len(results)
    print(f"\n✅ Found {result_count} result(s)")

    if result_count == 0:
        print("\n   No results found for this query.")
        print("\n" + "=" * 100 + "\n")
        return

    print("\n" + "-" * 100)

    intent = response.get('intent')

    # ---------------- ROUTE OPERATIONS ----------------
    if intent == "route_operations_query":
        print(f"\n{'#':<4} {'Flight':<10} {'Fleet Type':<20} {'Journeys':<10} {'Route':<20}")
        print("-" * 100)
        for i, r in enumerate(results[:10], 1):
            route = f"{r.get('origin', 'N/A')} → {r.get('destination', 'N/A')}"
            print(
                f"{i:<4} {r.get('flight_number', 'N/A'):<10} "
                f"{r.get('fleet_type', 'N/A'):<20} "
                f"{safe_int(r.get('journey_count')):<10} {route:<20}"
            )

    # ---------------- ROUTE DELAY ANALYSIS ----------------
    elif intent == "route_delay_analysis":
        print(f"\n{'#':<4} {'Flight':<10} {'Fleet Type':<20} {'Route':<15} {'Journeys':<10} {'Avg Delay':<12} {'On-time %':<12}")
        print("-" * 100)

        for i, r in enumerate(results, 1):
            route = f"{r.get('origin', 'N/A')} → {r.get('destination', 'N/A')}"
            print(
                f"{i:<4} {r.get('flight_number', 'N/A'):<10} "
                f"{r.get('fleet_type', 'N/A'):<20} {route:<15} "
                f"{safe_int(r.get('journey_count')):<10} "
                f"{safe_float(r.get('avg_delay')):<12.1f} "
                f"{safe_float(r.get('ontime_rate')):<12.1f}"
            )

        # Summary
        if len(results) > 1:
            total_journeys = sum(safe_int(r.get('journey_count')) for r in results)
            total_delayed = sum(safe_int(r.get('delayed_count')) for r in results)
            total_ontime = sum(safe_int(r.get('ontime_count')) for r in results)

            overall_avg = (
                sum(safe_float(r.get('avg_delay')) * safe_int(r.get('journey_count')) for r in results)
                / total_journeys if total_journeys > 0 else 0
            )

            print("-" * 100)
            print("\nRoute Summary:")
            print(f"  • Total Flights: {len(results)}")
            print(f"  • Total Journeys: {total_journeys}")
            print(f"  • Overall Avg Delay: {overall_avg:.1f} minutes")
            print(f"  • Delayed: {total_delayed} | On-time: {total_ontime}")
            if total_journeys > 0:
                print(f"  • Overall On-time Rate: {100 * total_ontime / total_journeys:.1f}%")

    # ---------------- ROUTE FREQUENCY ----------------
    elif intent == "route_frequency_ranking":
        print(f"\n{'Rank':<6} {'Route':<25} {'Journeys':<12}")
        print("-" * 100)
        for i, r in enumerate(results[:15], 1):
            route = f"{r.get('origin', 'N/A')} → {r.get('destination', 'N/A')}"
            print(f"{i:<6} {route:<25} {safe_int(r.get('journey_count')):<12}")

    # ---------------- AIRPORT TRAFFIC ----------------
    elif intent == "airport_traffic_analysis":
        print(f"\n{'Rank':<6} {'Airport':<12} {'Departures':<12} {'Arrivals':<12} {'Total Traffic':<15}")
        print("-" * 100)
        for i, r in enumerate(results[:15], 1):
            print(
                f"{i:<6} {r.get('airport_code', 'N/A'):<12} "
                f"{safe_int(r.get('departures')):<12} "
                f"{safe_int(r.get('arrivals')):<12} "
                f"{safe_int(r.get('total_traffic')):<15}"
            )

    # ---------------- FOOD SATISFACTION ----------------
    elif intent == "food_satisfaction_analysis":
        print(
        f"\n{'Category':<25} {'Journeys':<12} {'Avg Score':<12} "
        f"{'Min':<8} {'Max':<8} {'High Sat (≥7)':<15} {'Sat Rate %':<12}")
        print("-" * 100)

        for r in results:
          print(
            f"{r.get('group_by', 'N/A'):<25} "
            f"{safe_int(r.get('total_journeys')):<12} "
            f"{safe_float(r.get('avg_food_score')):<12.2f} "
            f"{safe_float(r.get('min_score')):<8.1f} "
            f"{safe_float(r.get('max_score')):<8.1f} "
            f"{safe_int(r.get('high_satisfaction_count')):<15} "
            f"{safe_float(r.get('satisfaction_rate')):<12.1f}"
        )

    # Overall summary (weighted)
        if results:
          total_journeys = sum(safe_int(r.get('total_journeys')) for r in results)

          weighted_avg = (
            sum(
                safe_float(r.get('avg_food_score')) * safe_int(r.get('total_journeys'))
                for r in results
            ) / total_journeys
            if total_journeys > 0 else 0
            )

        print("-" * 100)
        print("\nOverall Summary:")
        print(f"  • Total Journeys Analyzed: {total_journeys}")
        print(f"  • Weighted Average Food Score: {weighted_avg:.2f}/10")


    # ---------------- OVERALL SATISFACTION ----------------
    elif intent == "overall_satisfaction_analysis":
      for r in results:
        satisfied = safe_int(r.get('satisfied_passengers'))
        total_journeys = safe_int(r.get('total_journeys'))

        print("\n Overall Passenger Satisfaction (Weighted Formula):")
        print("   • Formula: 0.5×Food + 0.35×Delay + 0.1×Legs + 0.05×Miles")
        print("   • Threshold: Score > 3")
        print(f"   • Satisfied Passengers: {satisfied}")
        print(f"   • Total Journeys Analyzed: {total_journeys}")
        print(f"   • Average Satisfaction Score: {safe_float(r.get('avg_satisfaction')):.2f}/10")
        print(f"   • Min Score: {safe_float(r.get('min_satisfaction')):.2f}")
        print(f"   • Max Score: {safe_float(r.get('max_satisfaction')):.2f}")

        if satisfied > 0 and total_journeys > 0:
            rate = 100.0 * satisfied / total_journeys
            print(f"   • Satisfaction Rate: {rate:.1f}% of journeys")


    # ---------------- LOYALTY MILES ----------------
    elif intent == "loyalty_miles_analysis":
        print(f"\n{'Loyalty Level':<20} {'Total Miles':<15} {'Avg Miles':<15} {'Journeys':<12}")
        print("-" * 100)
        for r in results:
            print(
                f"{r.get('loyalty_level', 'N/A'):<20} "
                f"{safe_float(r.get('total_miles')):<15,.0f} "
                f"{safe_float(r.get('avg_miles')):<15,.0f} "
                f"{safe_int(r.get('journey_count')):<12}"
            )

    else:
        for i, r in enumerate(results[:10], 1):
            print(f"\n{i}. {json.dumps(r, indent=2)}")

    if result_count > 15:
        print(f"\n... and {result_count - 15} more results (showing first 15)")

    print("\n" + "=" * 100 + "\n")


In [ ]:
test_queries = [
        "What flights go from IAX to LAX?",

        "Show me routes with delays over 60 minutes from IAX to LAX",

        "What are the top 10 most frequent routes?",

        "Rank the 5 busiest airports",

        "What's the average food satisfaction score for economy class?",

        "How many premier gold members fly over 5000 miles?",

        "Show me B777-200 flights with delays",

        "How does gen-z rate economy class?",

        "Show me direct flight statistics",

        "What are the top 5 routes for Gen Z passengers in economy class with food scores above 7?"
    ]

for query in test_queries:
        response = process_user_query(
            user_query=query,
            neo4j_conn=driver
        )
        format_results(response)


📝 QUERY: What flights go from IAX to LAX?

🎯 Intent: route_operations_query

🔍 Extracted Entities:
   • origin_airport: IAX
   • destination_airport: LAX
   • all_airports: ['IAX', 'LAX']

✅ Found 10 result(s)

----------------------------------------------------------------------------------------------------

#    Flight     Fleet Type           Journeys   Route               
----------------------------------------------------------------------------------------------------
1    633        B737-900             4          IAX → LAX           
2    1828       B737-800             4          IAX → LAX           
3    797        B737-900             3          IAX → LAX           
4    1914       B737-MAX8            2          IAX → LAX           
5    1883       B737-900             2          IAX → LAX           
6    1878       B777-200             2          IAX → LAX           
7    2087       B737-900             1          IAX → LAX           
8    2026       B737-MAX9        

## Node Embeddings

We will use numerical feature
vectors derived from node properties.

The best node that contains the most numerical data is the **Journey** node.

## First Model Attempt

We will test out the following mapping model initially:


```
delay = min(abs(delay)/120, 1.0)
miles = min(miles/6000, 1.0)
legs = legs /4
```

and we will encode the passenger class for this journey.

In [ ]:
# Load csv first to know the all the available unique classes to encode.

df = pd.read_csv('/content/Airline_surveys_sample.csv')
unique_values = df['loyalty_program_level'].unique().tolist()

print(unique_values)

['non-elite', 'premier gold', 'premier silver', 'premier 1k', 'global services', 'premier platinum', 'NBK']


In [ ]:
LOYALTY_ENCODING = {
    "non-elite": 0,
    "premier silver": 1,
    "premier gold": 2,
    "premier platinum": 3,
    "premier 1k": 4,
    "global services": 5,
    "NBK": 6
}

We will fetch the existing journey nodes, embed them, then create the embedded nodes.

In [ ]:
FETCH_JOURNEYS_QUERY = """
MATCH (p:Passenger)-[:TOOK]->(j:Journey)
WHERE j.food_satisfaction_score IS NOT NULL
  AND j.arrival_delay_minutes IS NOT NULL
  AND j.actual_flown_miles IS NOT NULL
  AND j.number_of_legs IS NOT NULL
  AND p.loyalty_program_level IS NOT NULL
RETURN
    j.feedback_ID AS journey_id,
    j.food_satisfaction_score AS food,
    j.arrival_delay_minutes AS delay,
    j.actual_flown_miles AS miles,
    j.number_of_legs AS legs,
    p.loyalty_program_level AS loyalty
"""

def fetch_journeys(driver):
    with driver.session() as session:
        result = session.run(FETCH_JOURNEYS_QUERY)
        return [dict(r) for r in result]

In [ ]:
def build_journey_embedding(row):
    food_norm = row["food"] / 10.0
    delay_norm = min(abs(row["delay"]) / 120.0, 1.0)
    miles_norm = min(row["miles"] / 6000.0, 1.0)
    legs_norm = row["legs"] / 4.0

    loyalty_raw = LOYALTY_ENCODING.get(row["loyalty"].lower(), 0)
    loyalty_norm = loyalty_raw / 6.0

    return [
        round(food_norm, 4),
        round(delay_norm, 4),
        round(miles_norm, 4),
        round(legs_norm, 4),
        round(loyalty_norm, 4),
        loyalty_raw
    ]

In [ ]:
UPDATE_EMBEDDING_QUERY = """
MATCH (j:Journey {feedback_ID: $journey_id})
SET j.embedding = $embedding
"""
def write_embeddings(driver, journeys):
    with driver.session() as session:
        for row in journeys:
            embedding = build_journey_embedding(row)
            session.run(
                UPDATE_EMBEDDING_QUERY,
                journey_id=row["journey_id"],
                embedding=embedding
            )

In [ ]:
# journeys = fetch_journeys(driver)
# print(f"Fetched {len(journeys)} journeys")

# write_embeddings(driver, journeys)
# print("Journey embeddings written to Neo4j")


In [ ]:
# # Create the vector index
# CREATE_VECTOR_INDEX = """
# CREATE VECTOR INDEX journey_embedding_index
# IF NOT EXISTS
# FOR (j:Journey)
# ON (j.embedding)
# OPTIONS {
#   indexConfig: {
#     `vector.dimensions`: 6,
#     `vector.similarity_function`: 'cosine'
#   }
# }
# """
# with driver.session() as session:
#     session.run(CREATE_VECTOR_INDEX)

# print("Vector index created")


In [ ]:
# # Testing the existing embedding similarity search
# QUERY_VECTOR_SEARCH = """
# CALL db.index.vector.queryNodes(
#   'journey_embedding_index',
#   $k,
#   $query_vector
# )
# YIELD node AS j, score

# MATCH (p:Passenger)-[:TOOK]->(j)
# MATCH (j)-[:ON]->(f:Flight)
# MATCH (f)-[:DEPARTS_FROM]->(o:Airport)
# MATCH (f)-[:ARRIVES_AT]->(d:Airport)

# RETURN
#     j.feedback_ID AS journey_id,
#     j.food_satisfaction_score AS food,
#     j.actual_flown_miles AS miles,
#     j.arrival_delay_minutes AS delay,
#     p.loyalty_program_level AS loyalty,
#     f.flight_number AS flight_number,
#     f.fleet_type_description AS fleet_type,
#     o.station_code AS origin,
#     d.station_code AS destination,
#     score
# ORDER BY score DESC
# """


# def query_similar_journeys(driver, query_vector, k=5):
#     with driver.session() as session:
#         result = session.run(
#             QUERY_VECTOR_SEARCH,
#             query_vector=query_vector,
#             k=k
#         )
#         return [dict(r) for r in result]

# query_vector = [
#     0.9,    # high food satisfaction
#     0.1,    # low delay
#     0.8,    # long distance
#     0.25,   # one leg
#     1/6,    # premier gold
#     2
# ]

# results = query_similar_journeys(driver, query_vector)
# results

This is good for now. However, it seems as if we can use more data for relevance.

If the question was referencing mainly generation, the generation isn't embedded.
Moreover, for questions referencing mainly fleet type, satisfaction scores and so on, the embeddings are unsimilar.

So, we should add more features from different nodes in order to embed the Journey nodes. Upon fetching, we will fetch the node with all its relatives (all connected nodes) to populate our information.

## Feature Vector Embeddings

It feels like using text embeddings would be simpler for the purposes of this project.
Since using node embeddings at this point would require some sort of additional queries, and would require us to encode airport locations down, making the similarity search less efficient.

Moreover, using graphs at this point to capture various relationships that can be inferred from categorical variables in the csv, makes this feel like text embeddings are the smarter option at this point.

Let's make sure everything is going to work first before moving onto Langchain.

In [ ]:
df = pd.read_csv("/content/Airline_surveys_sample.csv")

def row_to_text(row):
    return (
        f"Flight number {row.flight_number}: "
        f"Journey from {row.origin_station_code} to {row.destination_station_code}, "
        f"Passenger {row.record_locator},"
        f"Passenger class {row.passenger_class}, "
        f"Loyalty level {row.loyalty_program_level}, "
        f"Food satisfaction score {row.food_satisfaction_score}, "
        f"Arrival delay {row.arrival_delay_minutes} minutes, "
        f"Number of legs {row.number_of_legs}, "
        f"Generation {row.generation},"
        f"Distance flown {row.actual_flown_miles} miles,"
    )

texts = df.apply(row_to_text, axis=1).tolist()



ids = df["feedback_ID"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

# Two different embedding models for comparison
model_1 = SentenceTransformer("all-MiniLM-L6-v2")
model_2 = SentenceTransformer("BAAI/bge-base-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings_1 = model_1.encode(texts, show_progress_bar=True)
embeddings_2 = model_2.encode(texts, show_progress_bar=True)
embeddings_1

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

array([[ 0.08779275, -0.06261972,  0.03479198, ...,  0.04484418,
        -0.08209413, -0.07384343],
       [ 0.09225228, -0.03622413,  0.02566866, ...,  0.08253823,
        -0.04503444, -0.07539122],
       [ 0.07023557,  0.04077817,  0.02121219, ..., -0.00147482,
        -0.12313656, -0.0295055 ],
       ...,
       [ 0.00835636, -0.02019492,  0.04716411, ...,  0.06844218,
        -0.10412946, -0.03949045],
       [ 0.0488788 ,  0.01740481,  0.02023799, ...,  0.00906327,
        -0.05559538, -0.0569341 ],
       [ 0.02683384, -0.05131021,  0.02507618, ...,  0.04040896,
        -0.07046557, -0.04656175]], dtype=float32)

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 29.8 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np

def build_l2_index(embeddings):
    d = embeddings.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(embeddings.astype("float32"))
    return index

index_l2_1 = build_l2_index(embeddings_1)
index_l2_2 = build_l2_index(embeddings_2)


In [ ]:
def build_cosine_index(embeddings):
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype("float32"))
    return index, embeddings

index_cos_1, emb_norm_1 = build_cosine_index(embeddings_1)
index_cos_2, emb_norm_2 = build_cosine_index(embeddings_2)

Testing cosine search:

In [ ]:
query = (
    "Long distance flight 1230 details "
    "for premium passengers"
)
q_emb_1 = model_1.encode([query])
q_emb_2 = model_2.encode([query])
q_emb_1_norm = q_emb_1 / np.linalg.norm(q_emb_1)
q_emb_2_norm = q_emb_2 / np.linalg.norm(q_emb_2)

def search(index, query_embedding, ids, texts, k=5):
    distances, indices = index.search(query_embedding.astype("float32"), k)
    results = []
    for i in indices[0]:
        results.append({
            "feedback_ID": ids[i],
            "text": texts[i]
        })
    return results

# results = search(index_cos_1, q_emb_1_norm, ids, texts)
results = search(index_cos_2, q_emb_2_norm, ids, texts)

for r in results:
    print(r["feedback_ID"], r["text"])

F_386 Flight number 1236: Journey from DEX to DFX, Passenger B5XX0R,Passenger class Economy, Loyalty level non-elite, Food satisfaction score 1, Arrival delay -13 minutes, Number of legs 1, Generation Gen X,Distance flown 641 miles,
F_2881 Flight number 1233: Journey from EWX to ORX, Passenger LFXXW9,Passenger class Economy, Loyalty level premier silver, Food satisfaction score 2, Arrival delay -12 minutes, Number of legs 1, Generation Millennial,Distance flown 719 miles,
F_1800 Flight number 920: Journey from LHX to IAX, Passenger C6XXLB,Passenger class Economy, Loyalty level premier gold, Food satisfaction score 3, Arrival delay 46 minutes, Number of legs 1, Generation Gen X,Distance flown 3677 miles,
F_695 Flight number 920: Journey from LHX to IAX, Passenger C3XXHM,Passenger class Economy, Loyalty level premier platinum, Food satisfaction score 3, Arrival delay 27 minutes, Number of legs 1, Generation Boomer,Distance flown 3677 miles,
F_2238 Flight number 1232: Journey from EWX to 

Testing euclidean distance search:

In [ ]:
query_embedding = model_1.encode([query])
query_embedding = np.asarray(query_embedding, dtype="float32")

query_embedding2 = model_2.encode([query])
query_embedding2 = np.asarray(query_embedding, dtype="float32")


# results = search(index_l2_1, query_embedding, ids, texts)
# for r in results:
#     print(r["feedback_ID"], r["text"])



results = search(index_l2_2, query_embedding2, ids, texts)
for r in results:
    print(r["feedback_ID"], r["text"])

F_255 Flight number 1237: Journey from EWX to PHX, Passenger K1XXSS,Passenger class Economy, Loyalty level non-elite, Food satisfaction score 4, Arrival delay -44 minutes, Number of legs 1, Generation Millennial,Distance flown 2133 miles,
F_2881 Flight number 1233: Journey from EWX to ORX, Passenger LFXXW9,Passenger class Economy, Loyalty level premier silver, Food satisfaction score 2, Arrival delay -12 minutes, Number of legs 1, Generation Millennial,Distance flown 719 miles,
F_1654 Flight number 1233: Journey from EWX to ORX, Passenger NLXXG6,Passenger class Economy, Loyalty level non-elite, Food satisfaction score 3, Arrival delay -9 minutes, Number of legs 3, Generation Boomer,Distance flown 719 miles,
F_1931 Flight number 1140: Journey from EWX to PBX, Passenger L9XX1Y,Passenger class Economy, Loyalty level non-elite, Food satisfaction score 2, Arrival delay -24 minutes, Number of legs 2, Generation Silent,Distance flown 1024 miles,
F_2968 Flight number 146: Journey from OPX to E

# LLM Layer

We now move onto langchain, where we will repeat our tokenization using all-MiniLM-L6-v2 withing LangChain, create a system prompt, and test out three different models using LangChain & OpenRouter.

In [ ]:
!pip install "langchain==0.1.16"
!pip install "langchain-community==0.0.30"
!pip install "langchain-huggingface==0.0.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.2
    Uninstalling tenacity-9.1.2:
      Successfully uninstalled tenacity-9.1.2
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: numpy
    Found exis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.0.38
    Uninstalling langchain-community-0.0.38:
      Successfully uninstalled langchain-community-0.0.38
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.16 requires langchain-community<0.1,>=0.0.32, but you have langchain-community 0.0.30 which is incompatible.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS as LCFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms.base import LLM
from langchain.schema import Document
from typing import Optional, List, Any
from pydantic import Field
import requests

In [ ]:
class OpenRouterLangChainWrapper(LLM):
    api_key: str = Field(...)
    model_name: str = Field(...)
    max_tokens: int = 500
    temperature: float = 0.1

    @property
    def _llm_type(self) -> str:
        return "openrouter_requests"

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": self.model_name,
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "max_tokens": self.max_tokens,
                "temperature": self.temperature
            }
        )

        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
llm_1 = OpenRouterLangChainWrapper(
    api_key=OPENROUTER_API_KEY,
    model_name="kwaipilot/kat-coder-pro:free"
)

In [ ]:
llm_2 = OpenRouterLangChainWrapper(
    api_key=OPENROUTER_API_KEY,
    model_name="google/gemma-3-27b-it:free"
)

In [ ]:
llm_3 = OpenRouterLangChainWrapper(
    api_key=OPENROUTER_API_KEY,
    model_name="qwen/qwen3-4b:free"
)

In [ ]:
import pandas as pd
df = pd.read_csv('/content/Airline_surveys_sample.csv')

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

documents = [
    Document(
        page_content=row_to_text(row),
        metadata={"passenger_id": row.record_locator}
    )
    for _, row in df.iterrows()
]


splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

documents = splitter.split_documents(documents)


vectorstore = LCFAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [ ]:
AIRLINE_PERSONA = (
    "You are a flight information assistant. "
    "You answer questions about airline journeys, routes, delays, "
    "passenger satisfaction, and operational flight data. "
    "You must rely ONLY on the provided context."
)

In [ ]:
from langchain.schema import Document

def kg_results_to_documents(kg_results):
    docs = []

    # Case 1: results already a list of dicts
    if isinstance(kg_results, list):
        rows = kg_results

    # Case 2: single dict (wrap it)
    elif isinstance(kg_results, dict):
        rows = [kg_results]

    # Case 3: anything else (string, None, error)
    else:
        return []

    for r in rows:
        if not isinstance(r, dict):
            continue  # extra safety

        text = "\n".join(f"{k}: {v}" for k, v in r.items())
        docs.append(
            Document(
                page_content=text,
                metadata={"source": "knowledge_graph"}
            )
        )

    return docs

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(text.lower().split())


def clean_and_merge_documents(kg_docs, vector_docs, max_docs=10):
    seen = set()
    merged = []

    # iterating and cleaning the text
    for doc in kg_docs + vector_docs:
        normalized = normalize_text(doc.page_content)
        h = hash(normalized)

        # removing duplicate documents
        if h not in seen:
            seen.add(h)
            merged.append(doc)

        if len(merged) >= max_docs:
            break

    return merged

In [ ]:
from langchain.prompts import PromptTemplate

QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Persona:
You are an Airline Company Flight Insights Assistant. You answer questions about airline journeys,
routes, delays, and passenger satisfaction. Use ONLY the provided context.

Context:
{context}

Task:
Answer the user's question using only the information above.
If the information is not present, say you do not have enough data.

User Question:
{question}
"""
)

In [ ]:
from langchain.schema import BaseRetriever
from typing import List

class StaticContextRetriever(BaseRetriever):
    documents: List[Document]

    def get_relevant_documents(self, query: str) -> List[Document]:
        return self.documents

    async def aget_relevant_documents(self, query: str) -> List[Document]:
        return self.documents

In [ ]:
from langchain.chains import RetrievalQA

def process_user_query(
    user_query,
    neo4j_conn,
    retriever,
    llm,
    classification_mode="llm"
):
    # 1. Intent classification
    if classification_mode == "llm":
        intent = classify_intent_llm(user_query)
    else:
        intent = classify_intent(user_query)

    # 2. Entity extraction
    entities = extract_all_entities(user_query)

    # 3. KG retrieval (baseline)
    kg_results = query_knowledge_graph(
        user_query, intent, entities, neo4j_conn
    )
    kg_docs = kg_results_to_documents(kg_results)
    print(kg_docs)
    # 4. FAISS semantic retrieval
    vector_docs = retriever.get_relevant_documents(user_query)

    # 5. Clean + merge results
    final_docs = clean_and_merge_documents(kg_docs, vector_docs)

    # 6. Create static retriever for RetrievalQA
    static_retriever = StaticContextRetriever(documents=final_docs)

    # 7. Build RetrievalQA chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=static_retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": QA_PROMPT},
        return_source_documents=True
    )

    # 8. Run RetrievalQA
    result = qa_chain({"query": user_query})

    return {
        "intent": intent,
        "entities": entities,
        "answer": result["result"],
        "context_used": result["source_documents"]
    }

In [ ]:
user_query = "Which flights from EWX to LAX have high food satisfaction and low arrival delays?"

print(process_user_query(user_query, driver, retriever, llm_1))

[Document(page_content="query: Which flights from EWX to LAX have high food satisfaction and low arrival delays?\nintent: route_operations_query\nentities: {'origin_airport': 'EWX', 'destination_airport': 'LAX', 'all_airports': ['EWX', 'LAX'], 'delay_threshold': None, 'miles_threshold': None, 'satisfaction_threshold': None, 'number_of_legs': None, 'top_n': None, 'passenger_class': None, 'loyalty_level': None, 'fleet_type': None, 'generation': None}\nparameters: {'origin': 'EWX', 'destination': 'LAX', 'top_n': 10, 'delay_threshold': None, 'miles_threshold': None, 'satisfaction_threshold': None, 'number_of_legs': None, 'passenger_class': None, 'loyalty_level': None, 'fleet_type': None, 'generation': None}\nresults: EagerResult(records=[<Record flight_number=1500 fleet_type='B787-10' journey_count=4 origin='EWX' destination='LAX'>, <Record flight_number=680 fleet_type='B737-900' journey_count=3 origin='EWX' destination='LAX'>, <Record flight_number=1526 fleet_type='B737-900' journey_count

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


{'intent': 'route_operations_query', 'entities': {'origin_airport': 'EWX', 'destination_airport': 'LAX', 'all_airports': ['EWX', 'LAX'], 'delay_threshold': None, 'miles_threshold': None, 'satisfaction_threshold': None, 'number_of_legs': None, 'top_n': None, 'passenger_class': None, 'loyalty_level': None, 'fleet_type': None, 'generation': None}, 'answer': 'Based on the provided data, the following flights from EWX to LAX have high food satisfaction (score 4 or 5) and low arrival delays (negative values indicate early arrivals):\n\n1. **Flight 598** - Food satisfaction: 5, Arrival delay: -15 minutes (early)\n2. **Flight 2204** - Food satisfaction: 5, Arrival delay: -29 minutes (early)\n3. **Flight 1154** - Food satisfaction: 5, Arrival delay: -1 minute (early)\n\nThese flights all have excellent food satisfaction scores and arrived early, meeting both criteria of high food satisfaction and low arrival delays.', 'context_used': [Document(page_content="query: Which flights from EWX to LAX 

# UI Layer

Creating backend file with all necessary Graph-RAG Components

In [2]:
%%writefile backend.py
import os
import re
import json
import requests
from functools import lru_cache
from typing import Optional, List, Dict, Any, Tuple
import time

import pandas as pd
from neo4j import GraphDatabase

import spacy
from pydantic import Field
from langchain.llms.base import LLM
from langchain.schema import Document, BaseRetriever
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS as LCFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# =========================
# CONFIG / CONSTANTS
# =========================

DEFAULT_CONFIG_PATH = "/content/ACL_MS3_Config.txt"
DEFAULT_CSV_PATH = "/content/Airline_surveys_sample.csv"

MODELS = [
    "kwaipilot/kat-coder-pro:free",
    "google/gemma-3-27b-it:free",
    "meta-llama/llama-3.3-70b-instruct:free",
]

EMBEDDING_MODELS = {
    "MiniLM-L6 (fast)": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small (best quality, but very slow)": "BAAI/bge-small-en-v1.5",
}

AIRPORT_CODE_PATTERN = re.compile(r"\b[A-Z]{3}\b")

QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Persona:
You are a flight information assistant. You answer questions about airline journeys,
routes, delays, and passenger satisfaction. Use ONLY the provided context.

Context:
{context}

Task:
Answer the user's question using only the information above.
If the information is not present, say you do not have enough data.

User Question:
{question}
"""
)

INTENT_PROMPT_TEMPLATE = """You are an intent classification assistant for an airline analytics system that helps analyze flight data, passenger satisfaction, and operational metrics.

Your task is to classify the user's query into exactly ONE of the following intents. Read each intent definition carefully and select the most appropriate match.

===== INTENT DEFINITIONS =====

1. route_operations_query
   PURPOSE: Find specific flights, routes, or operational details between two airports.
   USE WHEN: User wants to identify, list, or check flights connecting a specific origin to a specific destination.
   REQUIRES: Both origin and destination airports (e.g., "from LAX to JFK")
   EXAMPLES:
   - "What flights go from ORD to LAX?"
   - "Show me all routes between JFK and MIA"
   - "List Boeing 737 flights departing from SFO to DEN"
   - "How many economy class flights fly from ATL to BOS?"
   DO NOT USE FOR: Delay analysis, frequency rankings, or airport-wide statistics

2. route_delay_analysis
   PURPOSE: Analyze arrival delays and punctuality for specific routes.
   USE WHEN: User asks about delays, late/early arrivals, or on-time performance for flights between two specific airports.
   REQUIRES: Delay-related keywords (delay, late, early, punctual, on-time) AND route information
   EXAMPLES:
   - "What's the average delay from LAX to JFK?"
   - "Show me routes with delays over 60 minutes"
   - "Are flights between ORD and DFW usually on time?"
   - "Which routes have the worst punctuality?"
   DO NOT USE FOR: Airport-level delay statistics or counting flights

3. route_frequency_ranking
   PURPOSE: Rank routes by volume and identify most/least frequent routes.
   USE WHEN: User asks about which routes are most common, popular, or have the highest number of flights.
   FOCUSES ON: Route-level traffic comparison across the dataset
   EXAMPLES:
   - "What are the top 10 most frequent routes?"
   - "Which route has the most flights?"
   - "Show me the least traveled routes"
   - "Rank routes by number of journeys"
   DO NOT USE FOR: Airport-level traffic or specific route lookups

4. airport_traffic_analysis
   PURPOSE: Analyze airport-level traffic, hub importance, and airport rankings.
   USE WHEN: User asks about busiest airports, airport traffic volume, or which airports handle the most flights.
   FOCUSES ON: Airport-centric analysis (departures, arrivals, total traffic)
   EXAMPLES:
   - "Which airports have the most departures?"
   - "What are the top 5 busiest airports?"
   - "Show me airports ranked by total traffic"
   - "Which airport is the biggest hub?"
   DO NOT USE FOR: Route-specific queries or passenger satisfaction

5. food_satisfaction_analysis
   PURPOSE: Analyze in-flight food and meal service quality.
   USE WHEN: User asks specifically about food, meals, catering, or dining satisfaction.
   FOCUSES ON: Food satisfaction scores and meal quality metrics
   EXAMPLES:
   - "What's the average food score for business class?"
   - "How do passengers rate our meal service?"
   - "Show me food satisfaction by route"
   - "Which flights have the best catering reviews?"
   DO NOT USE FOR: Overall satisfaction (use overall_satisfaction_analysis instead)

6. overall_satisfaction_analysis
   PURPOSE: Compute composite passenger satisfaction using multiple factors.
   USE WHEN: User asks about overall satisfaction, happy passengers, or general experience quality.
   CONSIDERS: Multiple factors including food scores, delays, journey complexity, and distance
   EXAMPLES:
   - "How many passengers were satisfied overall?"
   - "What's the satisfaction rate for economy passengers?"
   - "Show me overall customer satisfaction trends"
   - "Which generation is most satisfied with our service?"
   DO NOT USE FOR: Food-only satisfaction (use food_satisfaction_analysis)

7. loyalty_miles_analysis
   PURPOSE: Analyze flight distance and mileage patterns by loyalty program tier.
   USE WHEN: User asks about miles flown, distance traveled, or mileage segmented by loyalty level.
   REQUIRES: Keywords about loyalty (premier, elite, member, loyalty level) AND mileage/distance
   EXAMPLES:
   - "How many miles do premier 1K members fly on average?"
   - "What's the total distance flown by gold members?"
   - "Show me average miles by loyalty level"
   - "Which loyalty tier travels the most?"
   DO NOT USE FOR: Satisfaction analysis or route frequency

8. fleet_performance_analysis
   PURPOSE: Analyze aircraft performance, utilization, and metrics by fleet type.
   USE WHEN: User asks about specific aircraft types, planes, or fleet comparison.
   REQUIRES: Keywords about aircraft (Boeing, Airbus, 737, A320, fleet, aircraft, plane)
   EXAMPLES:
   - "Which fleet has the best on-time performance?"
   - "How many Boeing 737 flights do we operate?"
   - "Show me satisfaction scores by aircraft type"
   - "What's the average delay for Airbus A320 flights?"
   DO NOT USE FOR: Route-specific queries without fleet context

9. passenger_segmentation_query
   PURPOSE: Analyze passenger demographics, behavior, and preferences by segment.
   USE WHEN: User asks about passenger types, generations, age groups, or demographic patterns.
   REQUIRES: Keywords about demographics (millennial, Gen Z, Gen X, baby boomer, generation)
   EXAMPLES:
   - "How do millennials rate our service?"
   - "What class do Gen Z passengers prefer?"
   - "Show me travel patterns for baby boomers"
   - "Which generation has the highest loyalty participation?"
   DO NOT USE FOR: General satisfaction without demographic context

10. journey_complexity_analysis
    PURPOSE: Analyze trip complexity, connections, and multi-leg journey patterns.
    USE WHEN: User asks about connections, layovers, direct flights, or journey segments.
    REQUIRES: Keywords about connections (layover, stopover, multi-leg, direct, non-stop, segments)
    EXAMPLES:
    - "How many direct flights do we offer?"
    - "What's the average number of connections for LAX to JFK?"
    - "Show me satisfaction for multi-leg journeys"
    - "Which routes require the most layovers?"
    DO NOT USE FOR: Simple route lookups without complexity focus

11. unknown
    PURPOSE: Catch-all for queries that don't match any intent.
    USE WHEN: The query is unclear, too vague, or doesn't relate to airline analytics.
    EXAMPLES:
    - "Hello, how are you?"
    - "What's the weather like?"
    - "Tell me a joke"
    - Gibberish or off-topic queries

===== CLASSIFICATION GUIDELINES =====

- If the query mentions TWO AIRPORTS with "from...to" or "between", consider route-specific intents (1, 2, or 3)
- If the query mentions "delay", "late", or "early" with a route, choose intent 2 (route_delay_analysis)
- If the query mentions "food" or "meal", choose intent 5 (food_satisfaction_analysis), NOT 6
- If the query mentions "satisfaction" without "food", choose intent 6 (overall_satisfaction_analysis)
- If the query mentions "busiest airports" or "most arrivals/departures", choose intent 4 (airport_traffic_analysis)
- If the query mentions "busiest routes" or "most frequent routes", choose intent 3 (route_frequency_ranking)
- If the query mentions loyalty levels AND miles/distance, choose intent 7 (loyalty_miles_analysis)
- If the query mentions aircraft/fleet types, choose intent 8 (fleet_performance_analysis)
- If the query mentions generations or demographics, choose intent 9 (passenger_segmentation_query)
- If the query mentions connections or layovers, choose intent 10 (journey_complexity_analysis)
- When in doubt between two intents, choose the MORE SPECIFIC one

===== YOUR TASK =====

User query: "{user_query}"

Analyze the query above and return ONLY the intent name from the list below. Do not include explanations, reasoning, or additional text.

Valid intents:
- route_operations_query
- route_delay_analysis
- route_frequency_ranking
- airport_traffic_analysis
- food_satisfaction_analysis
- overall_satisfaction_analysis
- loyalty_miles_analysis
- fleet_performance_analysis
- passenger_segmentation_query
- journey_complexity_analysis
- unknown

Your response (intent name only):"""


# =========================
# NEO4J
# =========================

def load_neo4j_config(config_path: str = DEFAULT_CONFIG_PATH) -> Dict[str, str]:
    config = {}
    with open(config_path, "r") as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                config[k] = v
    return config


def get_neo4j_driver(config_path: str = DEFAULT_CONFIG_PATH):
    cfg = load_neo4j_config(config_path)
    uri = cfg.get("NEO4J_URI", "neo4j://localhost:7687")
    user = cfg.get("NEO4J_USERNAME", "neo4j")
    pwd = cfg.get("NEO4J_PASSWORD", "password")
    return GraphDatabase.driver(uri, auth=(user, pwd))


def run_cypher(query: str, parameters: Optional[Dict[str, Any]] = None,
              config_path: str = DEFAULT_CONFIG_PATH) -> List[Dict[str, Any]]:
    driver = get_neo4j_driver(config_path)
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]


# =========================
# CYPHER TEMPLATES
# =========================

# ==================== CYPHER QUERY TEMPLATES ====================

CYPHER_TEMPLATES = {

    # Intent 1: Find flights between two airports
    "route_operations_query": """
        MATCH (origin:Airport {station_code: $origin})<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport {station_code: $destination})
        MATCH (j:Journey)-[:ON]->(f)
        RETURN f.flight_number as flight_number,
               f.fleet_type_description as fleet_type,
               COUNT(j) as journey_count,
               origin.station_code as origin,
               dest.station_code as destination
        ORDER BY journey_count DESC
        LIMIT $top_n
    """,

    # Intent 2: Analyze delays for routes (can be specific route or all routes)
    "route_delay_analysis": """
        MATCH (origin:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport)
        MATCH (j:Journey)-[:ON]->(f)
        WHERE j.arrival_delay_minutes IS NOT NULL
          AND ($origin IS NULL OR origin.station_code = $origin)
          AND ($destination IS NULL OR dest.station_code = $destination)
          AND ($delay_threshold IS NULL OR j.arrival_delay_minutes > $delay_threshold)
        WITH f, origin, dest,
             COUNT(j) as journey_count,
             AVG(j.arrival_delay_minutes) as avg_delay,
             MIN(j.arrival_delay_minutes) as min_delay,
             MAX(j.arrival_delay_minutes) as max_delay,
             SUM(CASE WHEN j.arrival_delay_minutes > 0 THEN 1 ELSE 0 END) as delayed_count,
             SUM(CASE WHEN j.arrival_delay_minutes <= 0 THEN 1 ELSE 0 END) as ontime_count
        RETURN f.flight_number as flight_number,
               f.fleet_type_description as fleet_type,
               origin.station_code as origin,
               dest.station_code as destination,
               journey_count,
               avg_delay,
               min_delay,
               max_delay,
               delayed_count,
               ontime_count,
               ROUND(100.0 * ontime_count / journey_count, 1) as ontime_rate
        ORDER BY avg_delay DESC
        LIMIT $top_n
    """,

    # Intent 3: Rank routes by frequency
    "route_frequency_ranking": """
        MATCH (origin:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport)
        MATCH (j:Journey)-[:ON]->(f)
        WHERE origin.station_code <> dest.station_code
        WITH origin.station_code as origin,
             dest.station_code as destination,
             COUNT(j) as journey_count
        ORDER BY journey_count DESC
        LIMIT $top_n
        RETURN origin, destination, journey_count
    """,

    # Intent 4: Analyze airport traffic volume
    "airport_traffic_analysis": """
        MATCH (a:Airport)
        OPTIONAL MATCH (a)<-[:DEPARTS_FROM]-(f1:Flight)<-[:ON]-(j1:Journey)
        OPTIONAL MATCH (a)<-[:ARRIVES_AT]-(f2:Flight)<-[:ON]-(j2:Journey)
        WITH a.station_code as airport_code,
             COUNT(DISTINCT j1) as departures,
             COUNT(DISTINCT j2) as arrivals,
             COUNT(DISTINCT j1) + COUNT(DISTINCT j2) as total_traffic
        WHERE total_traffic > 0
        ORDER BY total_traffic DESC
        LIMIT $top_n
        RETURN airport_code, departures, arrivals, total_traffic
    """,

    # Intent 5: Analyze food satisfaction (shows journeys matching criteria)
    "food_satisfaction_analysis": """
       MATCH (p:Passenger)-[:TOOK]->(j:Journey)-[:ON]->(f:Flight)
MATCH (f)-[:DEPARTS_FROM]->(origin:Airport)
MATCH (f)-[:ARRIVES_AT]->(dest:Airport)
WHERE j.food_satisfaction_score IS NOT NULL
  AND ($passenger_class IS NULL OR j.passenger_class = $passenger_class)
  AND ($satisfaction_threshold IS NULL OR j.food_satisfaction_score >= $satisfaction_threshold)
  AND ($loyalty_level IS NULL OR p.loyalty_program_level = $loyalty_level)
  AND ($generation IS NULL OR p.generation = $generation)
  AND ($origin IS NULL OR origin.station_code = $origin)
  AND ($destination IS NULL OR dest.station_code = $destination)

WITH
  j, p, origin, dest,
  CASE
    WHEN $origin IS NOT NULL AND $destination IS NOT NULL THEN origin.station_code + ' → ' + dest.station_code
    WHEN $passenger_class IS NOT NULL THEN j.passenger_class
    WHEN $generation IS NOT NULL THEN p.generation
    WHEN $loyalty_level IS NOT NULL THEN p.loyalty_program_level
    ELSE 'Overall'
  END AS group_by

WITH group_by,
     COUNT(j) AS total_journeys,
     AVG(j.food_satisfaction_score) AS avg_food_score,
     MIN(j.food_satisfaction_score) AS min_score,
     MAX(j.food_satisfaction_score) AS max_score,
     SUM(CASE WHEN j.food_satisfaction_score >= 3 THEN 1 ELSE 0 END) AS high_satisfaction_count

RETURN group_by,
       total_journeys,
       avg_food_score,
       min_score,
       max_score,
       high_satisfaction_count,
       ROUND(100.0 * high_satisfaction_count / total_journeys, 1) AS satisfaction_rate
ORDER BY avg_food_score DESC
LIMIT $top_n

    """,

    # Intent 6: Compute overall passenger satisfaction using weighted formula
   "overall_satisfaction_analysis": """
        MATCH (p:Passenger)-[:TOOK]->(j:Journey)
        WHERE j.food_satisfaction_score IS NOT NULL
          AND j.arrival_delay_minutes IS NOT NULL
          AND j.number_of_legs IS NOT NULL
          AND j.actual_flown_miles IS NOT NULL
        WITH p, j,
             CASE WHEN abs(j.arrival_delay_minutes)/20.0 > 5 THEN 5 ELSE abs(j.arrival_delay_minutes)/20.0 END AS delay_raw,
             CASE WHEN j.number_of_legs * 1.5 > 5 THEN 5 ELSE j.number_of_legs * 1.5 END AS legs_raw,
             CASE WHEN j.actual_flown_miles / 3000.0 > 5 THEN 5 ELSE j.actual_flown_miles / 3000.0 END AS miles_raw
        WITH p, j,
             5 - delay_raw AS delay_score,
             5 - (CASE WHEN j.number_of_legs * 1.5 > 5 THEN 5 ELSE j.number_of_legs * 1.5 END) AS legs_score,
             5 - (CASE WHEN j.actual_flown_miles / 3000.0 > 5 THEN 5 ELSE j.actual_flown_miles / 3000.0 END) AS miles_score
        WITH p, j,
             ROUND(0.5 * j.food_satisfaction_score +
                   0.35 * delay_score +
                   0.1 * legs_score +
                   0.05 * miles_score, 1) AS overall_satisfaction_score
        WHERE overall_satisfaction_score > 3
        WITH COUNT(DISTINCT p) as satisfied_passengers,
             AVG(overall_satisfaction_score) as avg_satisfaction,
             MIN(overall_satisfaction_score) as min_satisfaction,
             MAX(overall_satisfaction_score) as max_satisfaction
        RETURN satisfied_passengers,
               avg_satisfaction,
               min_satisfaction,
               max_satisfaction
    """,

    # Intent 7: Analyze miles by loyalty level
"loyalty_miles_analysis": """
        MATCH (p:Passenger)-[:TOOK]->(j:Journey)
        WHERE j.actual_flown_miles IS NOT NULL
          AND p.loyalty_program_level IS NOT NULL
        WITH p.loyalty_program_level as loyalty_level,
             SUM(j.actual_flown_miles) as total_miles,
             AVG(j.actual_flown_miles) as avg_miles,
             COUNT(j) as journey_count
        ORDER BY total_miles DESC
        RETURN loyalty_level, total_miles, avg_miles, journey_count
    """,

    # Intent 8: Analyze fleet performance
    "fleet_performance_analysis": """
        MATCH (f:Flight)
        MATCH (j:Journey)-[:ON]->(f)
        WHERE f.fleet_type_description IS NOT NULL
          AND j.arrival_delay_minutes IS NOT NULL
          AND ($fleet_type IS NULL OR f.fleet_type_description = $fleet_type)
        WITH f.fleet_type_description as fleet_type,
             COUNT(j) as total_flights,
             AVG(j.arrival_delay_minutes) as avg_delay,
             AVG(j.food_satisfaction_score) as avg_food_score,
             SUM(CASE WHEN j.arrival_delay_minutes <= 15 THEN 1 ELSE 0 END) as ontime_count
        RETURN fleet_type,
               total_flights,
               avg_delay,
               avg_food_score,
               ROUND(100.0 * ontime_count / total_flights, 2) as ontime_rate
        ORDER BY total_flights DESC
        LIMIT $top_n
    """,

    # Intent 9: Analyze passenger demographics
    "passenger_segmentation_query": """
        MATCH (p:Passenger)-[:TOOK]->(j:Journey)
        WHERE p.generation IS NOT NULL
          AND ($generation IS NULL OR p.generation = $generation)
          AND ($passenger_class IS NULL OR j.passenger_class = $passenger_class)
          AND ($satisfaction_threshold IS NULL OR j.food_satisfaction_score >= $satisfaction_threshold)
        WITH p.generation as generation,
             COUNT(DISTINCT p) as passenger_count,
             COUNT(j) as journey_count,
             AVG(j.food_satisfaction_score) as avg_food_satisfaction,
             AVG(j.actual_flown_miles) as avg_miles
        ORDER BY passenger_count DESC
        RETURN generation, passenger_count, journey_count, avg_food_satisfaction, avg_miles
    """,

    # Intent 10: Analyze journey complexity (connections/legs)
    "journey_complexity_analysis": """
        MATCH (j:Journey)-[:ON]->(f:Flight)
        WHERE j.number_of_legs IS NOT NULL
        WITH j.number_of_legs as legs,
             COUNT(j) as journey_count,
             AVG(j.food_satisfaction_score) as avg_satisfaction,
             AVG(j.arrival_delay_minutes) as avg_delay
        ORDER BY legs
        RETURN legs, journey_count, avg_satisfaction, avg_delay
    """
}


# =========================
# INTENT (LLM)
# =========================

def classify_intent_llm(user_query: str, model_name: str = "kwaipilot/kat-coder-pro:free") -> str:
    api_key = OPENROUTER_API_KEY
    if not api_key:
        return "unknown"

    prompt = INTENT_PROMPT_TEMPLATE.format(user_query=user_query)

    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        data=json.dumps({
            "model": model_name,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0
        })
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()


# =========================
# NLP / ENTITY EXTRACTION
# =========================

def get_nlp():
    return spacy.load("en_core_web_sm")


def load_valid_airports(csv_path: str = DEFAULT_CSV_PATH) -> set:
    df = pd.read_csv(csv_path)
    origin_codes = df["origin_station_code"].astype(str).str.upper()
    dest_codes = df["destination_station_code"].astype(str).str.upper()
    return set(origin_codes).union(set(dest_codes))


def load_valid_categories(csv_path: str = DEFAULT_CSV_PATH) -> Tuple[set, set]:
    df = pd.read_csv(csv_path)
    return set(df["passenger_class"].dropna()), set(df["loyalty_program_level"].dropna())


def load_valid_fleet_types(csv_path: str = DEFAULT_CSV_PATH) -> set:
    df = pd.read_csv(csv_path)
    return set(df["fleet_type_description"].dropna())


def load_valid_generations(csv_path: str = DEFAULT_CSV_PATH) -> set:
    df = pd.read_csv(csv_path)
    return set(df["generation"].dropna())


def extract_airports(text: str, valid_airports: set) -> List[str]:
    candidates = AIRPORT_CODE_PATTERN.findall(text.upper())
    return [c for c in candidates if c in valid_airports]


def extract_route(text: str, valid_airports: set) -> Tuple[Optional[str], Optional[str]]:
    airports = extract_airports(text, valid_airports)
    if len(airports) >= 2:
        return airports[0], airports[1]
    return None, None


def extract_delay_threshold(text: str) -> Optional[int]:
    nlp = get_nlp()
    doc = nlp(text.lower())
    txt = text.lower()

    for ent in doc.ents:
        if ent.label_ in {"CARDINAL", "QUANTITY", "TIME"}:
            nums = re.findall(r"\d+", ent.text)
            if not nums:
                continue
            value = int(nums[0])

            if any(k in txt for k in ["early", "earlier", "ahead"]):
                return -value
            if any(k in txt for k in ["delay", "late", "later", "behind"]):
                return value
            if any(k in txt for k in ["time", "minute", "hour"]):
                return value
    return None


def extract_miles_threshold(text: str) -> Optional[int]:
    nlp = get_nlp()
    doc = nlp(text.lower())
    for ent in doc.ents:
        if ent.label_ in {"CARDINAL", "QUANTITY"}:
            nums = re.findall(r"\d+", ent.text)
            if nums and any(k in text.lower() for k in ["mile", "km", "distance", "flown"]):
                return int(nums[0])
    return None


def extract_number_of_legs(text: str) -> Optional[int]:
    nlp = get_nlp()
    doc = nlp(text.lower())
    for ent in doc.ents:
        if ent.label_ == "CARDINAL":
            nums = re.findall(r"\d+", ent.text)
            if nums and any(k in text.lower() for k in ["leg", "segment", "stop", "connection", "layover"]):
                return int(nums[0])

    if any(k in text.lower() for k in ["direct", "nonstop", "non-stop"]):
        return 1
    return None


def extract_satisfaction_threshold(text: str) -> Optional[int]:
    nlp = get_nlp()
    doc = nlp(text.lower())
    for ent in doc.ents:
        if ent.label_ == "CARDINAL":
            nums = re.findall(r"\d+", ent.text)
            if nums and any(k in text.lower() for k in ["satisfaction", "score", "rating", "rated", "above", "below", "over", "under"]):
                return int(nums[0])
    return None


def extract_top_n(text: str) -> Optional[int]:
    nlp = get_nlp()
    doc = nlp(text.lower())
    txt = text.lower()
    ranking_keywords = ["top", "first", "best", "worst", "bottom", "least", "most", "rank"]

    for ent in doc.ents:
        if ent.label_ == "CARDINAL":
            nums = re.findall(r"\d+", ent.text)
            if not nums:
                continue
            for kw in ranking_keywords:
                if f"{kw} {nums[0]}" in txt or f"{nums[0]} {kw}" in txt:
                    return int(nums[0])

    if any(k in txt for k in ranking_keywords):
        return 10
    return None


def extract_passenger_class(text: str, valid_classes: set) -> Optional[str]:
    tl = text.lower()
    for cls in valid_classes:
        if str(cls).lower() in tl:
            return str(cls)
    return None


def extract_loyalty_level(text: str, valid_levels: set) -> Optional[str]:
    tl = text.lower()
    for lvl in valid_levels:
        if str(lvl).lower() in tl:
            return str(lvl)
    return None


def extract_fleet_type(text : str, valid_fleets: set) -> Optional[str]:
    tl = text.lower()
    for fleet in valid_fleets:
        if str(fleet).lower() in tl:
            return str(fleet)
    return None


def extract_generation(text: str, valid_generations: set) -> Optional[str]:
    tl = text.lower()
    for gen in valid_generations:
        if str(gen).lower() in tl:
            return str(gen)
    return None


def extract_all_entities(text: str, csv_path: str = DEFAULT_CSV_PATH) -> Dict[str, Any]:
    valid_airports = load_valid_airports(csv_path)
    valid_classes, valid_levels = load_valid_categories(csv_path)
    valid_fleets = load_valid_fleet_types(csv_path)
    valid_gens = load_valid_generations(csv_path)

    origin, destination = extract_route(text, valid_airports)

    return {
        "origin_airport": origin,
        "destination_airport": destination,
        "all_airports": extract_airports(text, valid_airports),
        "delay_threshold": extract_delay_threshold(text),
        "miles_threshold": extract_miles_threshold(text),
        "satisfaction_threshold": extract_satisfaction_threshold(text),
        "number_of_legs": extract_number_of_legs(text),
        "top_n": extract_top_n(text),
        "passenger_class": extract_passenger_class(text, valid_classes),
        "loyalty_level": extract_loyalty_level(text, valid_levels),
        "fleet_type": extract_fleet_type(text, valid_fleets),
        "generation": extract_generation(text, valid_gens),
    }

def evaluate_retrieval(
    retrieved_docs: List[Document],
    entities: Dict[str, Any],
    k: int
) -> Dict[str, float]:
    """
    Weakly-supervised retrieval evaluation.
    """
    relevant = 0

    entity_values = {
        str(v).lower()
        for v in entities.values()
        if v is not None
    }

    for doc in retrieved_docs[:k]:
        content = doc.page_content.lower()
        if any(val in content for val in entity_values):
            relevant += 1

    precision = relevant / k if k else 0.0

    return {
        "precision_at_k": round(precision, 3),
        "relevant_docs": relevant,
        "k": k
    }


# =========================
# KG QUERY WRAPPER
# =========================

def build_parameters(entities: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "origin": entities.get("origin_airport"),
        "destination": entities.get("destination_airport"),
        "top_n": entities.get("top_n") or 10,
        "delay_threshold": entities.get("delay_threshold"),
        "miles_threshold": entities.get("miles_threshold"),
        "satisfaction_threshold": entities.get("satisfaction_threshold"),
        "number_of_legs": entities.get("number_of_legs"),
        "passenger_class": entities.get("passenger_class"),
        "loyalty_level": entities.get("loyalty_level"),
        "fleet_type": entities.get("fleet_type"),
        "generation": entities.get("generation"),
    }


def query_knowledge_graph(user_query: str, intent: str, entities: Dict[str, Any],
                          config_path: str = DEFAULT_CONFIG_PATH) -> Dict[str, Any]:
    if intent == "unknown":
        return {"query": user_query, "intent": intent, "entities": entities, "cypher": None,
                "parameters": None, "results": None, "error": "Could not understand the query."}

    cypher_query = CYPHER_TEMPLATES.get(intent)
    if not cypher_query:
        return {"query": user_query, "intent": intent, "entities": entities, "cypher": None,
                "parameters": None, "results": None, "error": f"No template for intent: {intent}"}

    parameters = build_parameters(entities)

    try:
        results = run_cypher(cypher_query, parameters, config_path=config_path)
        return {
            "query": user_query,
            "intent": intent,
            "entities": entities,
            "cypher": cypher_query,
            "parameters": parameters,
            "results": results,
            "result_count": len(results),
            "error": None
        }
    except Exception as e:
        return {"query": user_query, "intent": intent, "entities": entities, "cypher": cypher_query,
                "parameters": parameters, "results": None, "error": f"Query failed: {e}"}


# =========================
# RAG (FAISS) BUILDERS
# =========================

def row_to_text(row) -> str:
    return (
        f"Flight number {row.flight_number}: "
        f"Journey from {row.origin_station_code} to {row.destination_station_code}, "
        f"Passenger {row.record_locator}, "
        f"Passenger class {row.passenger_class}, "
        f"Loyalty level {row.loyalty_program_level}, "
        f"Food satisfaction score {row.food_satisfaction_score}, "
        f"Arrival delay {row.arrival_delay_minutes} minutes, "
        f"Number of legs {row.number_of_legs}, "
        f"Generation {row.generation}, "
        f"Distance flown {row.actual_flown_miles} miles."
    )

def build_retriever(csv_path: str = DEFAULT_CSV_PATH, k: int = 10, embedding_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
  df = pd.read_csv(csv_path)
  embedding_model = HuggingFaceEmbeddings( model_name=embedding_name )
  docs = [ Document(page_content=row_to_text(row),
            metadata={"passenger_id": row.record_locator})
            for _, row in df.iterrows() ]
  splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
  docs = splitter.split_documents(docs)
  vectorstore = LCFAISS.from_documents(docs, embedding_model)
  return vectorstore.as_retriever(search_kwargs={"k": k})



# =========================
# LLM WRAPPER (OpenRouter)
# =========================

class OpenRouterLangChainWrapper(LLM):
    api_key: str = Field(...)
    model_name: str = Field(...)
    max_tokens: int = 500
    temperature: float = 0.1
    last_usage: Dict[str, int] = Field(default_factory=dict)

    @property
    def _llm_type(self) -> str:
        return "openrouter_requests"

    def _call(self, prompt: str, stop=None) -> str:
      resp = requests.post(
          "https://openrouter.ai/api/v1/chat/completions",
          headers={
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
          },
          json={
            "model": self.model_name,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": self.max_tokens,
            "temperature": self.temperature
          }
        )
      resp.raise_for_status()
      data = resp.json()

      self.last_usage = data.get("usage", {})
      return data["choices"][0]["message"]["content"]



def build_llm(model_name: str) -> OpenRouterLangChainWrapper:
    api_key = OPENROUTER_API_KEY
    return OpenRouterLangChainWrapper(api_key=api_key, model_name=model_name)


# =========================
# MERGING / DOCUMENT UTILS
# =========================

def kg_results_to_documents(kg_response: Dict[str, Any]) -> List[Document]:
    """
    kg_response is the dict returned by query_knowledge_graph().
    We convert only kg_response['results'] rows into Documents.
    """
    docs: List[Document] = []
    rows = kg_response.get("results") or []

    for r in rows:
        if not isinstance(r, dict):
            continue
        text = "\n".join(f"{k}: {v}" for k, v in r.items())
        docs.append(Document(page_content=text, metadata={"source": "knowledge_graph"}))
    return docs


def normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def clean_and_merge_documents(kg_docs: List[Document], vector_docs: List[Document], max_docs: int = 12) -> List[Document]:
    seen = set()
    merged: List[Document] = []
    for doc in (kg_docs + vector_docs):
        h = hash(normalize_text(doc.page_content))
        if h in seen:
            continue
        seen.add(h)
        merged.append(doc)
        if len(merged) >= max_docs:
            break
    return merged


class StaticContextRetriever(BaseRetriever):
    documents: List[Document]

    def get_relevant_documents(self, query: str) -> List[Document]:
        return self.documents

    async def aget_relevant_documents(self, query: str) -> List[Document]:
        return self.documents


# =========================
# MAIN PIPELINE (WHAT app.py CALLS)
# =========================

def answer_query(
    user_query: str,
    model_name: str,
    embedding_model_name : str,
    csv_path: str = DEFAULT_CSV_PATH,
    config_path: str = DEFAULT_CONFIG_PATH,
    classification_model: str = "kwaipilot/kat-coder-pro:free",
) -> Dict[str, Any]:
    """
    Returns everything needed for the Streamlit UI:
      - intent
      - entities
      - cypher_query
      - parameters
      - kg_raw_results
      - kg_docs / faiss_docs / merged_docs
      - final_answer
    """
    # 1) Intent + entities
    intent = classify_intent_llm(user_query, model_name=classification_model)
    entities = extract_all_entities(user_query, csv_path=csv_path)

    # 2) KG retrieval
    kg_response = query_knowledge_graph(user_query, intent, entities, config_path=config_path)
    kg_docs = kg_results_to_documents(kg_response)



    # 3) FAISS retrieval
    start_retrieval = time.perf_counter()
    retriever = build_retriever(
      csv_path=csv_path,
      embedding_name=embedding_model_name,
      k=10
    )
    vector_docs = retriever.get_relevant_documents(user_query)
    retrieval_time = (time.perf_counter() - start_retrieval) * 1000

    retrieval_metrics = evaluate_retrieval(
      retrieved_docs=vector_docs,
      entities=entities,
      k=10
    )



    # 4) Merge docs -> static retriever
    merged_docs = clean_and_merge_documents(kg_docs, vector_docs, max_docs=15)
    static_retriever = StaticContextRetriever(documents=merged_docs)

    # 5) LLM answer with RetrievalQA
    llm = build_llm(model_name)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=static_retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": QA_PROMPT},
        return_source_documents=True
    )

    start_llm = time.perf_counter()
    result = qa_chain({"query": user_query})
    llm_time = (time.perf_counter() - start_llm) * 1000


    return {
        "user_query": user_query,
        "intent": intent,
        "entities": entities,

        "cypher_query": kg_response.get("cypher"),
        "cypher_parameters": kg_response.get("parameters"),
        "kg_raw_results": kg_response.get("results"),
        "kg_error": kg_response.get("error"),

        "answer": result.get("result"),
        "source_documents": result.get("source_documents", []),

        "kg_docs": kg_docs,
        "faiss_docs": vector_docs,
        "merged_docs": merged_docs,

        "model_used": model_name,
        "metrics": {
        "retrieval": {
            **retrieval_metrics,
            "retrieval_time_ms": round(retrieval_time, 2)
        },
        "llm": {
            "model" : model_name,
            "llm_time_ms": round(llm_time, 2),
            "token_usage": llm.last_usage,
        },
        "end_to_end_time_ms": round(retrieval_time + llm_time, 2),
        "embedding_model": embedding_model_name,
        "llm_model": model_name
    }
    }


Overwriting backend.py


In [ ]:
import backend
out = backend.answer_query(
    "Which flights from EWX to LAX have high food satisfaction and low arrival delays?",
    model_name="google/gemma-3-27b-it:free"
)
out["intent"], out["cypher_query"], out["answer"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke in

('route_operations_query',
 '\n        MATCH (origin:Airport {station_code: $origin})<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(dest:Airport {station_code: $destination})\n        MATCH (j:Journey)-[:ON]->(f)\n        RETURN f.flight_number as flight_number, \n               f.fleet_type_description as fleet_type,\n               COUNT(j) as journey_count,\n               origin.station_code as origin,\n               dest.station_code as destination\n        ORDER BY journey_count DESC\n        LIMIT $top_n\n    ',
 '\nFlight number 598 has a food satisfaction score of 4 and an arrival delay of -8 minutes.\n\nI do not have food satisfaction or arrival delay data for any other flights from EWX to LAX.')

In [9]:
%%writefile app.py
import streamlit as st
import backend
import json

st.set_page_config(
    page_title="Airline Company Flight Insights Assistant",
    layout="wide"
)

# ======================
# SIDEBAR
# ======================

st.sidebar.title("Model Selection")

model_name = st.sidebar.selectbox(
    "Select LLM Model",
    backend.MODELS,
    index=1
)

st.sidebar.markdown("---")

embedding_label = st.sidebar.selectbox(
    "Select Embedding Model",
    list(backend.EMBEDDING_MODELS.keys()),
    index=0
)

embedding_model_name = backend.EMBEDDING_MODELS[embedding_label]


# ======================
# MAIN UI
# ======================

st.title("✈️ Airline Knowledge Graph Question Answering")

user_query = st.text_input(
    "Ask a question about flights, delays, satisfaction, routes, or passengers:",
    placeholder="Which flights from EWX to LAX have high food satisfaction and low arrival delays?"
)

run_button = st.button("🔍 Run Query")

if "model_metrics" not in st.session_state:
    st.session_state.model_metrics = []


# ======================
# RUN PIPELINE
# ======================

if run_button and user_query.strip():
    with st.spinner("Running analysis..."):
        result = backend.answer_query(
        user_query=user_query,
        model_name=model_name,
        embedding_model_name=embedding_model_name
      )
    st.session_state.model_metrics.append(result["metrics"])

    # ======================
    # FINAL ANSWER
    # ======================

    st.subheader("🧠 Final Answer")
    st.success(result["answer"])

    # ======================
    # INTENT & ENTITIES
    # ======================

    with st.expander("🎯 Detected Intent & Entities", expanded=False):
        st.markdown(f"**Intent:** `{result['intent']}`")
        st.json(result["entities"])

    # ======================
    # CYPHER QUERY
    # ======================

    with st.expander("🧬 Cypher Query Executed", expanded=False):
        if result["cypher_query"]:
            st.code(result["cypher_query"], language="cypher")
            st.markdown("**Parameters:**")
            st.json(result["cypher_parameters"])
        else:
            st.warning("No Cypher query executed.")

    # ======================
    # KG RAW RESULTS
    # ======================

    with st.expander("📊 Knowledge Graph Raw Results", expanded=False):
        if result["kg_raw_results"]:
            st.json(result["kg_raw_results"])
        else:
            st.warning("No results returned from Knowledge Graph.")

    # ======================
    # RETRIEVED CONTEXT
    # ======================

    with st.expander("📚 Retrieved Context Documents", expanded=False):
        for i, doc in enumerate(result["source_documents"], 1):
            source = doc.metadata.get("source", "faiss")
            st.markdown(f"**{i}. Source:** `{source}`")
            st.write(doc.page_content)
            st.markdown("---")

    import pandas as pd

    df = pd.DataFrame(st.session_state.model_metrics)
    st.dataframe(df)

else:
    st.info("Enter a question and click **Run Query** to begin.")


Overwriting app.py


In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.2 MB/s eta 0:00:00


In [4]:
from pyngrok import ngrok
ngrok.set_auth_token("2zYJ8mBaTPClXjwI0oAoEl3AElU_72PM6bT7cxvnyyEUP1QT5")

In [5]:
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &')

In [6]:
ngrok.kill()

In [7]:
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://a239b7b94b5f.ngrok-free.app" -> "http://localhost:8501">